# Сравнение детекторов аномалий траектории

## Коэффициент корреляции Мэттьюса (MCC)

В этом блокноте MCC — главный критерий выбора порога `τ` (и гиперпараметров вроде `eps`, `nu`) на обучающей выборке и один из критериев сравнения моделей на тесте. Считается **по точкам** (point-wise); положительный класс — **аномалия** (как в `confusion_rates`).

$$
\mathrm{MCC} = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}
$$

$TP$, $FP$, $FN$, $TN$ — число истинно/ложно положительных и истинно/ложно отрицательных предсказаний относительно эталонной маски `anom`.

**Основные свойства:**

- **Диапазон** $[-1,\,1]$: $1$ — идеальное совпадение разметок; $0$ — качество на уровне случайного угадывания при тех же долях классов; отрицательные значения — систематически хуже случайного.
- **Учитывает все четыре ячейки** таблицы сопряжённости, в отличие от accuracy при перекосе классов.
- **Симметричность по классам:** одинаково реагирует на ложные аномалии на штатном ходу ($FP$) и на пропуски аномалий ($FN$).
- **Устойчивость к дисбалансу** относительно доли положительного класса: тривиальные предсказания «всё anomaly» или «всё move» обычно дают низкий или нулевой MCC, даже если accuracy высока.
- **Не определён** (в коде принимается $0$), если в знаменателе ноль — например, когда предсказан только один класс или в выборке один класс.

После максимизации MCC среди кандидатов порога в полосе `MCC_BAND` выбирается меньший операционный distance loss (см. ниже).

Наблюдение — точка `(time, lat, lon)`; для обучения координаты переводятся в локальную ENU (м) через `LocalEnuConverter`. Задача бинарная: `anomaly` против `move`. Точки `stand` в выборку не входят. В `TrajectorySeries` метка `0` — аномалия, метка `1` — `move`.

ARIMA, GP, GP-EKF, LSTM, TCN, DONUT, Anomaly Transformer, STGVAD и One-Class SVM подгоняются по точкам `move`. HMM учится на отрезках обоих классов, Isolation Forest — по всей обучающей выборке. DBSCAN и LOF считают плотность размечаемого ряда. Порог или гиперпараметр (`eps`, `nu`, `v_max`) выбирается по обучающей выборке целиком. Главный критерий порога — коэффициент Мэттьюса (MCC): он одновременно штрафует и пропуск аномалий, и ложные тревоги на штатном ходу. Среди порогов в полосе `MCC_BAND` от лучшего MCC берётся меньший **операционный** distance loss (длина только по принятым move-точкам, без интерполяции в пропусках). Метрики на тесте считаются по последним 50 000 точкам. На тесте метрики считаются через `TrajectoryMetrics.compute_all` (классификация, Detection Delay, геопространственные метрики на заполненных траекториях).

Сравнение моделей на тесте идёт так же. Сначала MCC, при близких значениях — меньший distance loss, затем point-wise F1. Абсолютное положение в состоянии фильтра есть у Калмана, фильтра частиц и GP-EKF: проекция от медианы обучающего `move`, старт с первой точки своего ряда. GP внутри GP-EKF смотрит на скорость и шаг времени, не на координаты. У остальных детекторов признаки — шаг, скорость или форма окна, поэтому район карты сам по себе score не задаёт.



In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.special import logsumexp
from scipy.stats import chi2
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors
from sklearn.svm import OneClassSVM
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

SEED = 7
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

START_IND = 200_000
# В файле 419 453 строки
# С raw-индекса 315_185 идёт сплошная anomaly.
# тест и карта последних 50_000 точек содержат оба класса.
END_IND = 350_000
TEST_SIZE = 50_000
PLOT_TAIL = 50_000
# Среди порогов с MCC не хуже лучшего больше чем на эту величину берётся меньший distance loss.
MCC_BAND = 0.02
# Окно карты: [юг, запад, север, восток]. None — автоматический масштаб по точкам.
MAP_BBOX = [50.524974, 44.809115, 54.250411, 52.029355]
# Подложка OSM (нужен интернет). False — только трек на осях.
MAP_USE_BASEMAP = True
EARTH_RADIUS_M = 6_371_000.0
N_EPOCHS = 6
N_WINDOWS = 8_000
N_PARTICLES = 32


def find_root() -> Path:
    """Корень репозитория: рядом лежат data/2.csv и пакет app."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "2.csv").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Не найден корень проекта с data/2.csv")


path_root = find_root()
path = path_root / "data" / "2.csv"
sys.path.insert(0, str(path_root))

from app.evaluation import GeospatialMetrics, TrajectoryMetrics, TrajectorySeries
from app.viz import plot_track_maps

print(f"Корень: {path_root}")
print(f"Файл: {path}")
print(f"PyTorch {torch.__version__}, потоков {torch.get_num_threads()}")



# Данные

Из `../data/2.csv` берётся фрагмент `raw.iloc[START_IND:END_IND]`: `START_IND = 200_000`, `END_IND = 350_000`.


In [ ]:
raw_full = pd.read_csv(path)
raw = raw_full.iloc[START_IND:END_IND].reset_index(drop=True)

frame = raw.drop(columns=["sat", "is_water"]).copy()
frame["lat"] = pd.to_numeric(frame["lat"], errors="coerce")
frame["lon"] = pd.to_numeric(frame["lon"], errors="coerce")
keep = frame["lat"].notna() & frame["lon"].notna() & (frame["lat"] != 0.0) & (frame["lon"] != 0.0)
frame = frame.loc[keep].reset_index(drop=True)
status = frame["status"].astype(str)
n_stand = int((status == "stand").sum())
n_other = int((~status.isin(["move", "anomaly", "stand"])).sum())
frame = frame.loc[status.isin(["move", "anomaly"])].reset_index(drop=True)
order = np.argsort(pd.to_datetime(frame["time"]).to_numpy(), kind="mergesort")
frame = frame.iloc[order].reset_index(drop=True)

if len(frame) <= TEST_SIZE:
    raise RuntimeError(
        f"После очистки осталось {len(frame)} строк, этого мало для теста из {TEST_SIZE}."
    )

test_frame = frame.iloc[-TEST_SIZE:].reset_index(drop=True)
train_frame = frame.iloc[:-TEST_SIZE].reset_index(drop=True)


def nfmt(value):
    return f"{int(value):,}".replace(",", " ")


def make_pack(data):
    """Собирает одну выборку: время в секундах и бинарная метка anomaly / move."""
    stamp = pd.to_datetime(data["time"])
    status_values = data["status"].astype(str).to_numpy()
    anomaly = status_values == "anomaly"
    move = status_values == "move"
    if not np.all(anomaly | move):
        raise ValueError("В выборке остались точки не из классов move и anomaly.")
    return {
        "time": stamp.to_numpy(),
        "t": stamp.to_numpy().astype("datetime64[ns]").astype(np.int64) / 1e9,
        "lat": data["lat"].to_numpy(dtype=float),
        "lon": data["lon"].to_numpy(dtype=float),
        "anom": anomaly,
        "valid": move,
    }


def describe(name, pack):
    print(
        f"{name:<20} {nfmt(len(pack['anom'])):>10} точек   "
        f"аномалий {pack['anom'].mean():6.1%}   move {pack['valid'].mean():6.1%}"
    )


fit = make_pack(train_frame)
test = make_pack(test_frame)
if np.any(np.diff(fit["t"]) <= 0.0) or np.any(np.diff(test["t"]) <= 0.0):
    raise RuntimeError("Время в обучающей или тестовой выборке не строго возрастает.")
print(
    f"Файл целиком: {nfmt(len(raw_full))} строк, "
    f"фрагмент [{START_IND}:{END_IND}): {nfmt(len(raw))} строк"
)
print(
    f"Отброшено stand: {nfmt(n_stand)}. "
    f"Отброшено других меток: {nfmt(n_other)}. "
    f"Осталось move и anomaly: {nfmt(len(frame))}."
)
print("Признаки моделей: time, x/y (ENU, м), производные движения. Классы: anomaly и move.")
describe("Обучение", fit)
describe("Тест", test)
if not np.any(test["valid"]):
    print(
        "В тесте нет точек move. Эталонная длина маршрута на тесте тогда равна нулю, "
        "а distance loss — длине пути, который модель оставила классом move."
    )


# Признаки, порог и вывод метрик

## Общие признаки

По отфильтрованному ряду (без `stand`) для каждой точки $i>0$ задаются шаг во восток и север ($\Delta x_i$, $\Delta y_i$), геодезическая скорость $v_i$, ускорение $a_i$ и приращение курса $\Delta\psi_i$ (с нормировкой угла в $(-\pi,\pi]$). Форма окна длины $w=6$:

$$\mathrm{shape}_i = \frac{\sum_{k=i-w}^{i-1} d_{\mathrm{geo}}(p_k,p_{k+1})}{\max(d_{\mathrm{geo}}(p_{i-w},p_i),\,1)}, \qquad i>w; \quad \mathrm{shape}_i=1 \text{ иначе}.$$

Для фильтров в локальной проекции ENU опорная точка — медиана координат обучающего `move`; состояние инициализируется в первой точке обрабатываемого ряда.

## Подбор порога

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.

На тесте строятся `TrajectorySeries` (широта/долгота) и вызывается `TrajectoryMetrics.compute_all`. Геометрические $d_i$, RMSE, max и distance loss считаются в **локальной ENU** (`LocalEnuConverter`, `report/Отчёт.tex`). Операционный distance loss при подборе $\tau$ — длина цепочки `move` в метрах ENU.

Визуализация: `plot_track_maps` по последним `PLOT_TAIL` точкам теста, масштаб `MAP_BBOX`.


In [ ]:
from app.geo.local_enu import LocalEnuConverter


def meters_to_km(distance_m):
    """Перевод расстояния из метров в километры."""
    return float(distance_m) / 1000.0


def fmt_num(value, decimals=2):
    """Число; при |value| >= 1000 — разделитель тысяч «_»."""
    if value is None or not np.isfinite(float(value)):
        return "—"
    v = float(value)
    negative = v < 0
    v = abs(v)
    if v < 1000:
        text = f"{v:.{decimals}f}" if decimals else str(int(round(v)))
        if "." in text:
            text = text.rstrip("0").rstrip(".")
        return ("-" if negative else "") + text
    if decimals == 0:
        body = f"{int(round(v)):,}".replace(",", "_")
    else:
        whole = int(v)
        frac = v - whole
        frac_text = f"{frac:.{decimals}f}".split(".")[1].rstrip("0")
        body = f"{whole:,}".replace(",", "_")
        if frac_text:
            body = f"{body}.{frac_text}"
    return ("-" if negative else "") + body


def fmt_dl_m(loss_m):
    """Distance loss в километрах с единицей измерения."""
    return f"{fmt_num(meters_to_km(loss_m), 4)} км"


def fmt_distance_m(distance_m, decimals=2):
    """Расстояние в метрах с единицей измерения."""
    return f"{fmt_num(distance_m, decimals)} м"


def fmt_route_km(length_m, decimals=3):
    """Длина маршрута в километрах с единицей измерения."""
    return f"{fmt_num(meters_to_km(length_m), decimals)} км"


def format_metric_value(key, value):
    if value is None or not np.isfinite(value):
        return "—"
    if key in {"precision", "recall", "f1"}:
        v = float(value)
        return fmt_num(v, 4) if abs(v) >= 1000 else f"{v:.4f}"
    if key == "distance_loss_m":
        return fmt_num(meters_to_km(value), 4)
    if key in {"rmse_m", "max_deviation_m"}:
        return fmt_num(float(value), 2)
    if key.startswith("delay_") and not key.startswith("delay_n_"):
        v = float(value)
        return fmt_num(v, 4) if abs(v) >= 1000 else f"{v:.4f}"
    if key.startswith("delay_n_"):
        return fmt_num(int(value), 0)
    v = float(value)
    return fmt_num(v, 2) if abs(v) >= 1000 else f"{v:.2f}"



def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def haversine_m(lat1, lon1, lat2, lon2):
    """Геодезическое расстояние на сфере, в метрах."""
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    dlat = lat2 - lat1
    dlon = np.radians(np.asarray(lon2, dtype=float) - np.asarray(lon1, dtype=float))
    dlon = (dlon + np.pi) % (2.0 * np.pi) - np.pi
    step = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2.0 * EARTH_RADIUS_M * np.arcsin(np.sqrt(np.clip(step, 0.0, 1.0)))


def bearing(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    dlon = np.radians(lon2 - lon1)
    east = np.sin(dlon) * np.cos(lat2)
    north = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.arctan2(east, north)


def project_m(lat, lon, lat0, lon0):
    east = np.radians(lon - lon0) * np.cos(np.radians(lat0)) * EARTH_RADIUS_M
    north = np.radians(lat - lat0) * EARTH_RADIUS_M
    return np.column_stack([east, north])


def motion_features(x_m, y_m, t_sec):
    """Скорость, ускорение, поворот курса и шаг в локальных метрах. У первой точки нули."""
    count = len(x_m)
    dt = np.ones(count)
    dist = np.zeros(count)
    speed = np.zeros(count)
    acc = np.zeros(count)
    dpsi = np.zeros(count)
    dx = np.zeros(count)
    dy = np.zeros(count)
    if count < 2:
        return dt, dist, speed, acc, dpsi, dx, dy
    dt[1:] = np.diff(t_sec)
    dist[1:] = np.hypot(x_m[1:] - x_m[:-1], y_m[1:] - y_m[:-1])
    speed[1:] = dist[1:] / np.clip(dt[1:], 1.0, None)
    acc[1:] = np.diff(speed) / np.clip(dt[1:], 1.0, None)
    heading = np.zeros(count)
    dx_step = x_m[1:] - x_m[:-1]
    dy_step = y_m[1:] - y_m[:-1]
    heading[1:] = np.arctan2(dx_step, dy_step)
    if count > 2:
        dpsi[2:] = wrap_angle(np.diff(heading[1:]))
    # шаги уже в метрах ENU
    dx[1:] = x_m[1:] - x_m[:-1]
    dy[1:] = y_m[1:] - y_m[:-1]
    return dt, dist, speed, acc, dpsi, dx, dy


def track_shape(x_m, y_m, dist, window=6):
    """Отношение длины пути к смещению за окно. Прямой ход около 1, дрожание на месте больше."""
    count = len(dist)
    shape = np.ones(count)
    if count <= window:
        return shape
    cumulative = np.cumsum(dist)
    path = cumulative[window:] - cumulative[:-window]
    span = np.hypot(x_m[window:] - x_m[:-window], y_m[window:] - y_m[:-window])
    shape[window:] = path / np.maximum(span, 1.0)
    return shape


def add_motion(pack):
    dt, dist, speed, acc, dpsi, dx, dy = motion_features(pack["x"], pack["y"], pack["t"])
    pack.update(
        dt=dt,
        dist=dist,
        speed=speed,
        acc=acc,
        dpsi=dpsi,
        dx=dx,
        dy=dy,
        shape=track_shape(pack["x"], pack["y"], dist),
    )
    return pack


for pack in (fit, test):
    add_motion(pack)

ENU_CONVERTER = LocalEnuConverter.from_first_point(fit["lat"], fit["lon"])
for pack in (fit, test):
    pack["x"], pack["y"] = ENU_CONVERTER.to_local_xy(pack["lat"], pack["lon"])
    pack["xy"] = np.column_stack([pack["x"], pack["y"]])

normal_obs = np.column_stack([fit["speed"][fit["valid"]], fit["acc"][fit["valid"]], fit["dpsi"][fit["valid"]]])
obs_median = np.median(normal_obs, axis=0)
obs_mad = np.median(np.abs(normal_obs - obs_median), axis=0)
obs_mad = np.where(obs_mad < 1e-6, 1.0, obs_mad) * 1.4826


def kinematic_matrix(pack, sl=None):
    """(скорость, ускорение, поворот), нормированные по точкам move обучения."""
    raw_obs = np.column_stack([pack["speed"], pack["acc"], pack["dpsi"]])
    if sl is not None:
        raw_obs = raw_obs[sl]
    return np.clip((raw_obs - obs_median) / obs_mad, -25.0, 25.0)


def robust_params(matrix):
    center = np.median(matrix, axis=0)
    scale = np.subtract(*np.percentile(matrix, [75, 25], axis=0))
    scale = np.where(scale < 1e-6, 1.0, scale)
    return center, scale


def apply_robust(matrix, center, scale):
    return (matrix - center) / scale


def local_motion_matrix(pack):
    """Шаг, скорость и форма окна. Абсолютные координаты сюда не входят."""
    return np.column_stack(
        [
            pack["dx"],
            pack["dy"],
            pack["speed"],
            pack["acc"],
            pack["dpsi"],
            np.log(np.maximum(pack["shape"], 1.0)),
        ]
    )


def anomaly_f1(is_anomaly, predicted):
    true_positive = np.count_nonzero(is_anomaly & predicted)
    false_positive = np.count_nonzero(~is_anomaly & predicted)
    false_negative = np.count_nonzero(is_anomaly & ~predicted)
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    if precision + recall == 0.0:
        return 0.0
    return 2.0 * precision * recall / (precision + recall)


def confusion_rates(is_anomaly, predicted):
    """TPR, TNR, F1 и MCC. Положительный класс — аномалия."""
    is_anomaly = np.asarray(is_anomaly, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    true_positive = int(np.count_nonzero(is_anomaly & predicted))
    false_positive = int(np.count_nonzero(~is_anomaly & predicted))
    false_negative = int(np.count_nonzero(is_anomaly & ~predicted))
    true_negative = int(np.count_nonzero(~is_anomaly & ~predicted))
    tpr = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    tnr = true_negative / (true_negative + false_positive) if true_negative + false_positive else 0.0
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    f1 = 2.0 * precision * tpr / (precision + tpr) if precision + tpr else 0.0
    denominator = (
        (true_positive + false_positive)
        * (true_positive + false_negative)
        * (true_negative + false_positive)
        * (true_negative + false_negative)
    )
    if denominator == 0:
        mcc = 0.0
    else:
        mcc = (true_positive * true_negative - false_positive * false_negative) / np.sqrt(denominator)
    return {"tpr": float(tpr), "tnr": float(tnr), "f1": float(f1), "mcc": float(mcc)}


def remember_train_operating_point(distance_loss, f1, mcc):
    """Запоминает distance loss, F1 и MCC порога на всей обучающей выборке."""
    global LAST_TRAIN_DL, LAST_TRAIN_F1, LAST_TRAIN_MCC
    LAST_TRAIN_DL = float(distance_loss)
    LAST_TRAIN_F1 = float(f1)
    LAST_TRAIN_MCC = float(mcc)


def operating_record(pack, predicted, tau=0.0):
    """MCC, F1 и distance loss одной маски аномалий."""
    predicted = np.asarray(predicted, dtype=bool)
    loss, ratio = route_length_pair(pack, predicted)
    record = confusion_rates(pack["anom"], predicted)
    record.update(tau=float(tau), loss=float(loss), ratio=float(ratio))
    return record


def operational_path_length_m(x_m, y_m, move_mask):
    """Длина ломаной по принятым move-точкам (без интерполяции в пропусках)."""
    x_arr = np.asarray(x_m, dtype=np.float64)
    y_arr = np.asarray(y_m, dtype=np.float64)
    move = np.asarray(move_mask, dtype=bool)
    x_k = x_arr[move]
    y_k = y_arr[move]
    if x_k.size < 2:
        return 0.0
    return float(np.sum(np.hypot(np.diff(x_k), np.diff(y_k))))


def reference_route_length(pack):
    """Операционная длина эталона: цепочка GT move-точек. Кэш route_ref_operational_m."""
    cached = pack.get("route_ref_operational_m")
    if cached is not None:
        return float(cached)
    move = ~np.asarray(pack["anom"], dtype=bool)
    length = operational_path_length_m(pack["x"], pack["y"], move)
    pack["route_ref_operational_m"] = float(length)
    return pack["route_ref_operational_m"]


def route_length_pair(pack, predicted_anomaly):
    """Операционный distance loss для подбора порога (без интерполяции)."""
    ref = reference_route_length(pack)
    predicted_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    move = ~predicted_anomaly
    predicted_length = operational_path_length_m(pack["x"], pack["y"], move)
    loss = abs(predicted_length - ref)
    ratio = loss / ref if ref > 0.0 else float("nan")
    return float(loss), float(ratio)


def report_reference_route_length(pack):
    """Эталонная длина для отчётности (заполнение аномалий по Отчёт.tex)."""
    cached = pack.get("route_ref_report_m")
    if cached is not None:
        return float(cached)
    truth = np.where(pack["anom"], 0.0, 1.0)
    ground_truth = TrajectorySeries(pack["time"], pack["lon"], pack["lat"], truth)
    length = GeospatialMetrics.reference_path_length_m(ground_truth)
    pack["route_ref_report_m"] = float(length)
    return pack["route_ref_report_m"]


def pick_operating_point(records):
    """Максимум MCC. В полосе MCC_BAND — минимум distance loss."""
    usable = [
        record
        for record in records
        if np.isfinite(record["loss"]) and np.isfinite(record["mcc"])
    ]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора порога.")
    best_mcc = max(record["mcc"] for record in usable)
    band = [record for record in usable if record["mcc"] >= best_mcc - MCC_BAND - 1e-12]
    band.sort(key=lambda record: (record["loss"], -record["mcc"], -record["f1"], float(record.get("tau", 0.0))))
    return band[0], float(best_mcc), len(band)


def select_threshold(scores, pack, extra=()):
    """Ищет tau по квантилям score всей обучающей выборки.

    Большой score — более аномальная точка. Сначала максимизируется MCC.
    Среди порогов в полосе MCC_BAND берётся минимальный distance loss.
    """
    scores = np.nan_to_num(np.asarray(scores, dtype=float), nan=0.0, posinf=1e12, neginf=0.0)
    is_anomaly = np.asarray(pack["anom"], dtype=bool)
    if len(scores) != len(is_anomaly):
        raise ValueError("Длина score не совпадает с обучающей выборкой.")
    quantiles = np.quantile(scores, np.linspace(0.01, 0.995, 180))
    extras = np.atleast_1d(np.asarray(extra, dtype=float))
    candidates = np.unique(np.concatenate([quantiles, extras])) if extras.size else quantiles
    records = [operating_record(pack, scores > tau, tau) for tau in candidates]
    chosen, best_mcc, band_size = pick_operating_point(records)
    normal = scores[~is_anomaly]
    anomalous = scores[is_anomaly]
    if len(normal) and len(anomalous):
        print(f"  медиана score: move {np.median(normal):.4g}, аномалии {np.median(anomalous):.4g}")
    print(
        f"  выбран порог {chosen['tau']:.4g}: MCC {chosen['mcc']:.3f}, "
        f"лучший MCC {best_mcc:.3f}, в полосе {band_size}, "
        f"DL {fmt_dl_m(chosen['loss'])}, F1 {chosen['f1']:.3f}, "
        f"доля move сохранена {chosen['tnr']:.1%}"
    )
    remember_train_operating_point(chosen["loss"], chosen["f1"], chosen["mcc"])
    return chosen["tau"], chosen["f1"]


def series_pair(predicted_anomaly):
    """Собирает TrajectorySeries prediction и ground truth на тесте. Метка 0 — аномалия."""
    predicted_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    truth = np.where(test["anom"], 0.0, 1.0)
    predicted = np.where(predicted_anomaly, 0.0, 1.0)
    ground_truth = TrajectorySeries(test["time"], test["lon"], test["lat"], truth)
    prediction = TrajectorySeries(
        test["time"], test["lon"].copy(), test["lat"].copy(), predicted
    )
    return prediction, ground_truth


def official_metrics(predicted_anomaly, anomaly_scores=None):
    """Считает метрики TrajectoryMetrics на тестовой выборке."""
    prediction, ground_truth = series_pair(predicted_anomaly)
    metrics = TrajectoryMetrics.compute_all(prediction, ground_truth)
    ref_report = report_reference_route_length(test)
    metrics["distance_loss_ratio"] = (
        float(metrics["distance_loss_m"] / ref_report) if ref_report > 0.0 else float("nan")
    )
    return metrics


RESULTS = []
LAST_TRAIN_DL = float("nan")
LAST_TRAIN_F1 = float("nan")
LAST_TRAIN_MCC = float("nan")
RATE_FIELDS = [
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1"),
]
LIST_FIELDS = [
    ("distance_loss_m", "Distance loss, км"),
    ("distance_loss_ratio", "Distance loss ratio"),
] + RATE_FIELDS + [
    ("delay_median", "Delay median, с"),
    ("delay_mean", "Delay mean, с"),
    ("delay_p95", "Delay p95, с"),
    ("delay_max", "Delay max, с"),
    ("delay_n_on_time", "On-time episodes"),
    ("delay_n_late_or_missed", "Late/missed episodes"),
    ("rmse_m", "RMSE, м"),
    ("max_deviation_m", "Max(d_i), м"),
]
COUNT_KEYS = {
    "delay_n_on_time",
    "delay_n_late_or_missed",
}


def format_metric(key, value):
    if value is None:
        return f"{'—':>14}"
    if key in COUNT_KEYS:
        return f"{int(value):>14d}"
    if not np.isfinite(value):
        return f"{'—':>14}"
    return f"{format_metric_value(key, value):>14}"


def plot_metric_strip(name, metrics):
    labels = [label for _, label in RATE_FIELDS]
    values = []
    for key, _ in RATE_FIELDS:
        value = metrics[key]
        values.append(np.nan if not np.isfinite(value) else float(value))
    fig, ax = plt.subplots(figsize=(11.2, 1.45))
    image = np.ma.masked_invalid(np.asarray(values, dtype=float)[None, :])
    heat = ax.imshow(image, cmap="YlOrRd", vmin=0.0, vmax=1.0, aspect="auto")
    ax.set_yticks([0])
    ax.set_yticklabels([name])
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right")
    for index, value in enumerate(values):
        text = "—" if not np.isfinite(value) else f"{value:.3f}"
        ax.text(index, 0, text, ha="center", va="center", color="black", fontsize=9)
    fig.colorbar(heat, ax=ax, fraction=0.02, pad=0.02)
    fig.tight_layout()
    try:
        from IPython import get_ipython

        interactive = get_ipython() is not None
    except Exception:
        interactive = False
    if interactive:
        plt.show()
    else:
        plt.close(fig)


def slice_series(series, tail):
    """Берёт хвост TrajectorySeries для карты."""
    return TrajectorySeries(
        series.time[-tail:],
        series.lon[-tail:],
        series.lat[-tail:],
        series.label[-tail:],
    )


def plot_classification_map(name, predicted_anomaly):
    """Карта ground truth и разметки модели по последним PLOT_TAIL точкам теста."""
    tail = min(PLOT_TAIL, len(predicted_anomaly))
    prediction, ground_truth = series_pair(predicted_anomaly)
    prediction = slice_series(prediction, tail)
    ground_truth = slice_series(ground_truth, tail)
    print(f"  Карта классификации: последние {nfmt(tail)} точек теста")
    map_kwargs = {"use_basemap": MAP_USE_BASEMAP}
    if MAP_BBOX is not None:
        map_kwargs["bbox"] = MAP_BBOX
    plot_track_maps(
        prediction,
        ground_truth,
        title=f"{name}: последние {nfmt(tail)} точек теста",
        figsize=(22.0, 11.0),
        annotate=False,
        **map_kwargs,
    )


def show_result(name, predicted_anomaly, elapsed, note, anomaly_scores=None):
    """Печатает метрики TrajectoryMetrics, рисует полоску, карту и запоминает строку сводки."""
    global RESULTS, LAST_TRAIN_DL, LAST_TRAIN_F1, LAST_TRAIN_MCC
    predicted_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    if predicted_anomaly.shape != test["anom"].shape:
        raise ValueError(f"{name}: длина маски {predicted_anomaly.shape[0]} вместо {test['anom'].shape[0]}")
    if anomaly_scores is not None and len(anomaly_scores) != len(predicted_anomaly):
        raise ValueError(f"{name}: длина score {len(anomaly_scores)} вместо {len(predicted_anomaly)}")
    metrics = official_metrics(predicted_anomaly, anomaly_scores)
    print("=" * 72)
    print(name)
    if note:
        print(note)
    print(f"  Помечено аномалиями на тесте: {predicted_anomaly.mean():.1%}   время {elapsed:.1f} с")
    print("  TrajectoryMetrics, положительный класс — аномалия (метка 0)")
    for key, label in LIST_FIELDS:
        print(f"    {label:<28} {format_metric(key, metrics[key]).strip():>14}")
    test_rates = confusion_rates(test["anom"], predicted_anomaly)
    print(f"    {'MCC':<28} {test_rates['mcc']:14.4f}")
    print(f"    {'Доля move сохранена':<28} {test_rates['tnr']:14.1%}")
    plot_metric_strip(name, metrics)
    plot_classification_map(name, predicted_anomaly)
    row = {
        "Метод": name,
        "Время, с": elapsed,
        "DL обучения, км": meters_to_km(LAST_TRAIN_DL),
        "F1 обучения": LAST_TRAIN_F1,
        "MCC обучения": LAST_TRAIN_MCC,
        "mcc": test_rates["mcc"],
    }
    for key, _ in LIST_FIELDS:
        row[key] = metrics[key]
    RESULTS = [item for item in RESULTS if item["Метод"] != name]
    RESULTS.append(row)
    LAST_TRAIN_DL = float("nan")
    LAST_TRAIN_F1 = float("nan")
    LAST_TRAIN_MCC = float("nan")


def window_starts(pack, length, max_dt, max_dist, require_normal):
    """Индексы окон, которые не пересекают большой разрыв по времени или дистанции."""
    count = len(pack["lat"])
    if count < length:
        return np.empty(0, dtype=int)
    good = np.ones(count, dtype=bool)
    if require_normal:
        good &= pack["valid"]
    step_ok = np.ones(count, dtype=bool)
    step_ok[1:] = (pack["dt"][1:] <= max_dt) & (pack["dist"][1:] <= max_dist)
    good_cum = np.cumsum(good.astype(np.int32))
    step_cum = np.cumsum(step_ok.astype(np.int32))
    starts = np.arange(count - length + 1)
    good_sum = good_cum[starts + length - 1] - np.where(starts > 0, good_cum[starts - 1], 0)
    step_sum = step_cum[starts + length - 1] - step_cum[starts]
    return starts[(good_sum == length) & (step_sum == length - 1)]


def take_starts(starts, limit):
    if len(starts) <= limit:
        return starts
    picked = rng.choice(len(starts), size=limit, replace=False)
    return np.sort(starts[picked])


def scatter_max(count, starts, values, length):
    """Раскладывает оконные score по точкам, в пересечениях оставляет максимум."""
    scores = np.full(count, -np.inf)
    index = starts[:, None] + np.arange(length)[None, :]
    if np.ndim(values) == 1:
        payload = np.repeat(values, length)
    else:
        payload = np.asarray(values, dtype=float).ravel()
    np.maximum.at(scores, index.ravel(), payload)
    scores[~np.isfinite(scores)] = 0.0
    return scores


def train_torch(model, array, loss_fn, epochs, lr=1e-3, batch_size=256):
    model.to(DEVICE)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(np.ascontiguousarray(array))), batch_size=batch_size, shuffle=True
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch)
            seen += len(batch)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


def batched_numpy(model, array, forward, batch_size=512):
    model.eval()
    pieces = []
    with torch.no_grad():
        for start in range(0, len(array), batch_size):
            batch = torch.from_numpy(np.ascontiguousarray(array[start : start + batch_size])).to(DEVICE)
            pieces.append(forward(model, batch).detach().cpu().numpy())
    return np.concatenate(pieces, axis=0)


origin_ecef = ENU_CONVERTER.origin_ecef
print(
    f"ENU: опора lat/lon = {fit['lat'][0]:.6f}, {fit['lon'][0]:.6f}; "
    f"ECEF origin = ({origin_ecef[0]:.1f}, {origin_ecef[1]:.1f}, {origin_ecef[2]:.1f}) м"
)
print(
    f"Операционный эталон (цепочка move): обучение {fmt_route_km(reference_route_length(fit))}, "
    f"тест {fmt_route_km(reference_route_length(test))}. "
    f"Отчётный эталон на тесте (с интерполяцией): {fmt_route_km(report_reference_route_length(test))}. "
    f"Полоса MCC — {MCC_BAND:.2f}: внутри неё операционный distance loss."
)


# Опорные разметки

Две постоянные маски задают края шкалы accuracy на этом тесте: ноль, если все точки объявлены штатными, и единица, если все объявлены аномальными.


In [ ]:
started = time.perf_counter()
show_result(
    "Всегда штатная",
    np.zeros(len(test["anom"]), dtype=bool),
    time.perf_counter() - started,
    "Каждая тестовая точка помечена штатной.",
)
started = time.perf_counter()
show_result(
    "Всегда аномалия",
    np.ones(len(test["anom"]), dtype=bool),
    time.perf_counter() - started,
    "Каждая тестовая точка помечена аномальной.",
)


# ARIMA(2, 1, 1)

## Принцип

Модель прогнозирует следующее приращение координат по истории; большая геодезическая ошибка одношагового прогноза указывает на аномалию.

## Формализация

Для каждой координаты плоскости ENU ($x_t$, $y_t$, метры) используется ARIMA$(2,1,1)$:

$$\phi(B)(1-B)x_t = c + \theta(B)\varepsilon_t, \qquad \Delta x_t = c + \phi_1\Delta x_{t-1} + \phi_2\Delta x_{t-2} + \varepsilon_t + \theta_1\varepsilon_{t-1}.$$

Прогноз $\hat{x}_t$ обновляется по фактическому приращению. Остаток $r_t = x_t - \hat{x}_t$. Score точки (после объединения широты и долготы):

$$s_i = d_{\mathrm{geo}}\bigl((\mathrm{lat}_i,\mathrm{lon}_i),(\hat{\mathrm{lat}}_i,\hat{\mathrm{lon}}_i)\bigr).$$

**Оценка параметров:** коэффициенты $(c,\phi_1,\phi_2,\theta_1)$ для широты и долготы — регрессия по лагам разностей (метод наименьших квадратов) по самому длинному непрерывному участку обучающего `move` без больших разрывов по времени и расстоянию.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def longest_smooth_slice(pack, max_dt=40.0, max_dist=2_000.0, min_len=80):
    good = pack["valid"].copy()
    good[1:] &= (pack["dt"][1:] <= max_dt) & (pack["dist"][1:] <= max_dist)
    padded = np.r_[False, good, False]
    change = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    if len(starts) == 0:
        raise RuntimeError("Нет гладкого штатного участка для ARIMA.")
    lengths = ends - starts
    best = int(np.argmax(lengths))
    if lengths[best] < min_len:
        raise RuntimeError("Самый длинный штатный участок короче 80 точек.")
    return int(starts[best]), int(ends[best])


def fit_arima_211(series):
    """Оценивает ARIMA(2, 1, 1): константу, два AR-коэффициента разностей и один MA."""
    series = np.asarray(series, dtype=float)
    delta = np.diff(series)
    constant = float(np.mean(delta))
    centered = delta - constant
    long_order = 12
    rows = len(centered) - long_order
    design = np.column_stack([centered[long_order - lag : len(centered) - lag] for lag in range(1, long_order + 1)])
    target = centered[long_order:]
    coef, *_ = np.linalg.lstsq(design, target, rcond=None)
    residual = np.zeros(len(centered))
    residual[long_order:] = target - design @ coef
    start = long_order + 1
    target2 = centered[start:]
    design2 = np.column_stack(
        [
            centered[start - 1 : len(centered) - 1],
            centered[start - 2 : len(centered) - 2],
            residual[start - 1 : len(centered) - 1],
        ]
    )
    beta, *_ = np.linalg.lstsq(design2, target2, rcond=None)
    phi1, phi2, theta = (float(item) for item in beta)
    while abs(phi1) > 1.8 or abs(phi2) > 0.95:
        phi1 *= 0.8
        phi2 *= 0.8
    return {"c": constant, "phi1": phi1, "phi2": phi2, "theta": theta}


def arima_forecast(series, params):
    """Одношаговый прогноз ARIMA(2, 1, 1) с обновлением по фактическому приращению."""
    predicted = np.empty(len(series))
    predicted[0] = series[0]
    prev_delta_1 = 0.0
    prev_delta_2 = 0.0
    prev_eps = 0.0
    for index in range(1, len(series)):
        delta_hat = (
            params["c"] + params["phi1"] * prev_delta_1 + params["phi2"] * prev_delta_2 + params["theta"] * prev_eps
        )
        predicted[index] = series[index - 1] + delta_hat
        delta = series[index] - series[index - 1]
        prev_eps = delta - delta_hat
        prev_delta_2 = prev_delta_1
        prev_delta_1 = delta
    return predicted


def arima_scores(pack, x_params, y_params):
    x_hat = arima_forecast(pack["x"], x_params)
    y_hat = arima_forecast(pack["y"], y_params)
    return np.hypot(pack["x"] - x_hat, pack["y"] - y_hat)


print("ARIMA(2, 1, 1)")
started = time.perf_counter()
span_start, span_end = longest_smooth_slice(fit)
print(f"  участок обучения: точки {span_start}–{span_end}, длина {span_end - span_start}")
x_params = fit_arima_211(fit["x"][span_start:span_end])
y_params = fit_arima_211(fit["y"][span_start:span_end])
print(
    f"  ENU x   c={x_params['c']:.3g} φ1={x_params['phi1']:.3f} φ2={x_params['phi2']:.3f} θ={x_params['theta']:.3f}"
)
print(
    f"  ENU y   c={y_params['c']:.3g} φ1={y_params['phi1']:.3f} φ2={y_params['phi2']:.3f} θ={y_params['theta']:.3f}"
)
fit_scores = arima_scores(fit, x_params, y_params)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = arima_scores(test, x_params, y_params)
print(f"  порог {fmt_distance_m(tau)}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "ARIMA(2, 1, 1)",
    test_scores > tau,
    time.perf_counter() - started,
    f"Порог геодезической ошибки {fmt_distance_m(tau, 1)}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Фильтр Калмана, постоянная скорость

## Принцип

Скрытое состояние — положение и скорость в ENU; измерение — координаты. Аномалия — измерение, слабо согласованное с предсказанием фильтра.

## Формализация

$$\mathbf{x}_i = (p_x,p_y,v_x,v_y)^\top, \quad \mathbf{x}_i = A(\Delta t_i)\mathbf{x}_{i-1} + \mathbf{w}_i, \quad \mathbf{z}_i = H\mathbf{x}_i + \mathbf{v}_i, \quad H=\begin{pmatrix}1&0&0&0\\0&1&0&0\end{pmatrix}.$$

Инновация $\mathbf{r}_i = \mathbf{z}_i - H\hat{\mathbf{x}}_{i|i-1}$, ковариация $S_i$. Нормированный квадрат инновации (NIS):

$$s_i = \mathbf{r}_i^\top S_i^{-1}\mathbf{r}_i.$$

При гауссовской модели $s_i\sim\chi^2_2$. Шум процесса — дискретное белое ускорение; $\Delta t_i$ в модели ограничивается сверху. При $\Delta t_i>180$ с состояние переинициализируется по измерению, $s_i$ текущего шага сохраняется.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.
В перебор кандидатов $\tau$ добавляется квантиль $\chi^2_{0{,}999}(2)$ как ориентир.


In [ ]:
def process_noise(dt, sigma_acc):
    dt2 = dt * dt
    dt3 = dt2 * dt
    dt4 = dt2 * dt2
    noise = np.zeros((4, 4))
    noise[0, 0] = noise[1, 1] = dt4 / 4.0
    noise[2, 2] = noise[3, 3] = dt2
    noise[0, 2] = noise[2, 0] = noise[1, 3] = noise[3, 1] = dt3 / 2.0
    return noise * (sigma_acc**2)


def kalman_scores(pack, sigma_acc=0.35, sigma_meas=40.0):
    xy = pack["xy"]
    t_sec = pack["t"]
    count = len(xy)
    state = np.zeros(4)
    state[:2] = xy[0]
    cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    observation = np.array([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0]])
    meas_cov = np.eye(2) * (sigma_meas**2)
    eye = np.eye(4)
    scores = np.zeros(count)
    for index in range(1, count):
        dt = float(t_sec[index] - t_sec[index - 1])
        dt = 1.0 if dt <= 0.0 else dt
        gap = dt > 180.0
        dt_model = min(dt, 45.0)
        transition = np.eye(4)
        transition[0, 2] = dt_model
        transition[1, 3] = dt_model
        pred = transition @ state
        pred_cov = transition @ cov @ transition.T + process_noise(dt_model, sigma_acc)
        innov = xy[index] - observation @ pred
        innov_cov = observation @ pred_cov @ observation.T + meas_cov
        solved = np.linalg.solve(innov_cov, innov)
        scores[index] = float(innov @ solved)
        gain = pred_cov @ observation.T @ np.linalg.inv(innov_cov)
        state = pred + gain @ innov
        cov = (eye - gain @ observation) @ pred_cov
        cov = 0.5 * (cov + cov.T)
        if gap:
            state = np.array([xy[index, 0], xy[index, 1], 0.0, 0.0])
            cov = np.diag([80.0**2, 80.0**2, 5.0**2, 5.0**2])
    return scores


print("Фильтр Калмана")
started = time.perf_counter()
chi_critical = float(chi2.ppf(0.999, 2))
fit_scores = kalman_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit, extra=(chi_critical,))
test_scores = kalman_scores(test)
print(f"  χ² 0.999 (df=2) = {chi_critical:.2f}; выбранный порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "Фильтр Калмана",
    test_scores > tau,
    time.perf_counter() - started,
    f"Порог NIS {tau:.2f}. Теоретический χ²₀.₉₉₉ = {chi_critical:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Bootstrap particle filter (SIR)

## Принцип

Вместо одного гауссовского состояния поддерживается набор гипотез о движении; аномалия — измерение с низкой правдоподобностью относительно ансамбля.

## Формализация

Аппроксимация $p(\mathbf{x}_i\mid \mathbf{z}_{1:i}) \approx \sum_{j=1}^{K} w_i^{(j)}\delta(\mathbf{x}_i-\mathbf{x}_i^{(j)})$. Шаг SIR: ресэмплирование, прогноз частиц, веса $\tilde w_i^{(j)} \propto p(\mathbf{z}_i\mid \mathbf{x}_i^{(j)})$. Score:

$$s_i = -\log\left(\frac{1}{K}\sum_{j=1}^{K} p(\mathbf{z}_i\mid \mathbf{x}_i^{(j)})\right).$$

Модель движения — постоянная скорость в ENU с теми же ограничениями $\Delta t$, что у фильтра Калмана. При большом разрыве времени или $s_i>40$ частицы переинициализируются в точке измерения.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def systematic_resample(log_weights, generator):
    shifted = np.exp(log_weights - np.max(log_weights))
    weights = shifted / shifted.sum()
    positions = (np.arange(len(weights)) + generator.random()) / len(weights)
    cumulative = np.cumsum(weights)
    cumulative[-1] = 1.0
    return np.searchsorted(cumulative, positions, side="left").clip(0, len(weights) - 1)


def particle_scores(pack, n_particles=N_PARTICLES, sigma_acc=0.45, sigma_meas=40.0, seed=SEED):
    generator = np.random.default_rng(seed)
    xy = pack["xy"]
    t_sec = pack["t"]
    count = len(xy)
    particles = np.zeros((n_particles, 4))
    particles[:, :2] = xy[0]
    particles[:, 2:] = generator.normal(0.0, 1.0, size=(n_particles, 2))
    scores = np.zeros(count)
    r_var = sigma_meas**2
    log_const = np.log(2.0 * np.pi * r_var)
    for index in range(1, count):
        dt = float(t_sec[index] - t_sec[index - 1])
        dt = 1.0 if dt <= 0.0 else dt
        gap = dt > 180.0
        dt_model = min(dt, 45.0)
        particles[:, 0] += particles[:, 2] * dt_model
        particles[:, 1] += particles[:, 3] * dt_model
        particles[:, 0] += generator.normal(0.0, sigma_acc * dt_model**2 / 2.0, n_particles)
        particles[:, 1] += generator.normal(0.0, sigma_acc * dt_model**2 / 2.0, n_particles)
        particles[:, 2] += generator.normal(0.0, sigma_acc * dt_model, n_particles)
        particles[:, 3] += generator.normal(0.0, sigma_acc * dt_model, n_particles)
        diff = xy[index] - particles[:, :2]
        loglik = -0.5 * (np.sum(diff * diff, axis=1) / r_var + 2.0 * log_const)
        scores[index] = float(-logsumexp(loglik) + np.log(n_particles))
        draw = systematic_resample(loglik, generator)
        particles = particles[draw]
        if gap or scores[index] > 40.0:
            particles[:, :2] = xy[index]
            particles[:, 2:] = generator.normal(0.0, 1.0, size=(n_particles, 2))
    return scores


print("Particle filter")
started = time.perf_counter()
fit_scores = particle_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = particle_scores(test)
print(f"  частиц {N_PARTICLES}, порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "Particle filter",
    test_scores > tau,
    time.perf_counter() - started,
    f"SIR, частиц {N_PARTICLES}. Порог {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Gaussian process

## Принцип

Штатные приращения координат во времени моделируются гауссовским процессом; точка аномальна, если фактический шаг сильно отклоняется от прогноза с учётом неопределённости.

## Формализация

Независимые GP для $\Delta x(t)$ и $\Delta y(t)$ с ковариацией RBF. Прогноз $f(t_*)\mid\mathcal{D} \sim \mathcal{N}(\mu_*,\sigma_*^2)$. Score:

$$s_i = \frac{(\Delta x_i-\mu_x(t_i))^2}{\sigma_x^2(t_i)} + \frac{(\Delta y_i-\mu_y(t_i))^2}{\sigma_y^2(t_i)}.$$

**Обучение:** опорные моменты времени — подвыборка точек `move` обучения; гиперпараметры длины корреляции и шума наблюдения задаются в коде (сетка не используется). При удалении от опорных точек прогноз сходится к среднему шага, поэтому смена района карты сама по себе не повышает $s_i$.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def rbf(left, right, lengthscale, signal):
    gap = left.reshape(-1, 1) - right.reshape(1, -1)
    return (signal**2) * np.exp(-0.5 * gap * gap / lengthscale**2)


def gp_predict(t_train, y_train, t_test, lengthscale, noise, signal):
    center = float(np.mean(y_train))
    target = y_train - center
    chol = None
    jitter = 1e-4 * signal**2
    for _ in range(6):
        gram = rbf(t_train, t_train, lengthscale, signal)
        gram.flat[:: len(t_train) + 1] += noise**2 + jitter
        try:
            chol = np.linalg.cholesky(gram)
            break
        except np.linalg.LinAlgError:
            jitter *= 10.0
    if chol is None:
        raise np.linalg.LinAlgError("Ковариация GP не разложилась.")
    alpha = np.linalg.solve(chol.T, np.linalg.solve(chol, target))
    mean = np.empty(len(t_test))
    variance = np.empty(len(t_test))
    batch = 2_500
    for start in range(0, len(t_test), batch):
        chunk = t_test[start : start + batch]
        cross = rbf(t_train, chunk, lengthscale, signal)
        mean[start : start + len(chunk)] = cross.T @ alpha + center
        solved = np.linalg.solve(chol, cross)
        variance[start : start + len(chunk)] = signal**2 + noise**2 - np.sum(solved * solved, axis=0)
    return mean, np.clip(variance, 1e-6, None)


def gp_scores(pack, t_train, x_train, y_train, lengthscale, noise):
    signal_x = float(np.std(x_train) + 1.0)
    signal_y = float(np.std(y_train) + 1.0)
    mean_x, var_x = gp_predict(t_train, x_train, pack["t"], lengthscale, noise, signal_x)
    mean_y, var_y = gp_predict(t_train, y_train, pack["t"], lengthscale, noise, signal_y)
    return (pack["dx"] - mean_x) ** 2 / var_x + (pack["dy"] - mean_y) ** 2 / var_y


print("Gaussian process")
started = time.perf_counter()
normal_index = np.flatnonzero(fit["valid"])
gp_count = 250
grid = np.linspace(0, len(normal_index) - 1, gp_count).astype(int)
chosen = normal_index[grid]
t_gp = fit["t"][chosen]
x_gp = fit["dx"][chosen]
y_gp = fit["dy"][chosen]
lengthscale = 6.0 * 3600.0
step_scale = float(np.median(np.hypot(x_gp, y_gp)))
noise = max(5.0, 0.25 * step_scale)
fit_scores = gp_scores(fit, t_gp, x_gp, y_gp, lengthscale, noise)
tau, train_f1 = select_threshold(fit_scores, fit, extra=(chi2.ppf(0.999, 2),))
test_scores = gp_scores(test, t_gp, x_gp, y_gp, lengthscale, noise)
print(
    f"  опорных точек {gp_count}, длина корреляции {lengthscale / 3600:.1f} ч, шум шага {noise:.1f} м, "
    f"порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}"
)
show_result(
    "Gaussian process",
    test_scores > tau,
    time.perf_counter() - started,
    f"RBF по шагу dx, dy, опорных точек move {gp_count}, ℓ = 6 ч. Порог {tau:.2f}. "
    f"DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)






# GP-EKF

## Принцип

Расширенный фильтр Калмана: параметрический переход «постоянная скорость» дополняется остатком, обученным гауссовским процессом по скорости и шагу времени; наблюдение — линейное по положению (ГНСС).

## Формализация

Состояние $\mathbf{x}_i=(E,N,v^E,v^N)^\top$. Параметрический прирост $f(\mathbf{x},\Delta t)$ — смещение на $\Delta t$ при постоянной скорости. GP задаёт распределение остатка перехода; в шаге EKF якобиан среднего GP входит в линеаризацию. Инновация и NIS как у фильтра Калмана: $s_i=\mathbf{r}_i^\top S_i^{-1}\mathbf{r}_i$.

**Оптимизация:** гиперпараметры ядра GP (длина корреляции, шум) — по максимуму маргинального правдоподобия на штатных переходах обучения (равномерная подвыборка опорных переходов, не более 160).

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
from app.detectors.gp_bayes_filter import GpEkfModel

print("GP-EKF")
started = time.perf_counter()
gp_ekf = GpEkfModel(support=160, seed=SEED, sigma_meas=40.0).fit(fit["xy"], fit["t"], fit["valid"])
fit_scores = gp_ekf.scores(fit["xy"], fit["t"])
tau, train_f1 = select_threshold(fit_scores, fit, extra=(chi2.ppf(0.999, 2),))
test_scores = gp_ekf.scores(test["xy"], test["t"])
print(
    f"  опорных переходов {gp_ekf.n_support}, длина корреляции {gp_ekf.lengthscale:.2g}, шум {gp_ekf.noise:.2g}, "
    f"порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}"
)
show_result(
    "GP-EKF",
    test_scores > tau,
    time.perf_counter() - started,
    (
        f"GP-BayesFilters, остаток постоянной скорости. Опорных переходов {gp_ekf.n_support}, "
        f"длина корреляции {gp_ekf.lengthscale:.2g}, шум {gp_ekf.noise:.2g}. Порог NIS {tau:.2f}. "
        f"DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}."
    ),
    test_scores,
)


# Скрытая марковская модель

## Принцип

Наблюдение — вектор кинематики $(v,a,\Delta\psi)$ после робастной шкалы. Отдельные гауссовские HMM обучаются на штатных и аномальных отрезках; аномалия — низкая правдоподобность под штатной моделью относительно аномальных.

## Формализация

Скрытое состояние $S_i\in\{1,\ldots,K\}$, эмиссии $o_i\mid S_i=k \sim \mathcal{N}(\mu_k,\mathrm{diag}(\sigma_k^2))$. Предсказательное лог-правдоподобие $\log P(o_i\mid o_{1:i-1},\theta)$. Для моделей $\theta_N$ (штат), $\theta_A$, опционально $\theta_F$ (быстрый аномальный режим):

$$s_i = \max_{k\in\{A,F\}} \log P(o_i\mid o_{1:i-1},\theta_k) - \log P(o_i\mid o_{1:i-1},\theta_N).$$

**Оптимизация:** параметры HMM — итерации EM на последовательностях соответствующего класса (фиксированное число итераций в реализации).

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
class DiagHMM:
    """HMM с диагональными гауссовскими эмиссиями и несколькими последовательностями."""

    def __init__(self, n_states=3, n_iter=8, seed=SEED):
        self.n_states = n_states
        self.n_iter = n_iter
        self.rng = np.random.default_rng(seed)

    def _log_emit(self, values):
        diff = values[:, None, :] - self.mean[None, :, :]
        quad = np.sum(diff * diff / self.var[None, :, :], axis=-1)
        logdet = np.sum(np.log(2.0 * np.pi * self.var), axis=-1)
        return -0.5 * (quad + logdet)

    def _forward(self, log_emit):
        length, _ = log_emit.shape
        log_alpha = np.empty_like(log_emit)
        log_trans = np.log(self.trans + 1e-16)
        log_alpha[0] = np.log(self.start + 1e-16) + log_emit[0]
        for index in range(1, length):
            log_alpha[index] = log_emit[index] + logsumexp(log_alpha[index - 1][:, None] + log_trans, axis=0)
        return log_alpha

    def _backward(self, log_emit):
        length, _ = log_emit.shape
        log_beta = np.zeros_like(log_emit)
        log_trans = np.log(self.trans + 1e-16)
        for index in range(length - 2, -1, -1):
            log_beta[index] = logsumexp(
                log_trans + log_emit[index + 1][None, :] + log_beta[index + 1][None, :],
                axis=1,
            )
        return log_beta

    def fit(self, sequences):
        stacked = np.concatenate(sequences, axis=0)
        width = stacked.shape[1]
        levels = np.linspace(0.2, 0.8, self.n_states)
        self.mean = np.quantile(stacked, levels, axis=0)
        self.mean = self.mean + self.rng.normal(0.0, 0.01, size=self.mean.shape)
        base_var = np.var(stacked, axis=0) + 1e-2
        self.var = np.repeat(base_var[None, :], self.n_states, axis=0)
        self.trans = np.full((self.n_states, self.n_states), 1.0 / self.n_states)
        self.start = np.full(self.n_states, 1.0 / self.n_states)
        for _ in range(self.n_iter):
            start_acc = np.zeros(self.n_states)
            trans_acc = np.zeros_like(self.trans)
            mean_acc = np.zeros((self.n_states, width))
            var_acc = np.zeros((self.n_states, width))
            weight = np.zeros(self.n_states)
            for seq in sequences:
                log_emit = self._log_emit(seq)
                log_alpha = self._forward(log_emit)
                log_beta = self._backward(log_emit)
                log_gamma = log_alpha + log_beta
                log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
                gamma = np.exp(log_gamma)
                log_xi = (
                    log_alpha[:-1, :, None]
                    + np.log(self.trans + 1e-16)[None, :, :]
                    + log_emit[1:, None, :]
                    + log_beta[1:, None, :]
                )
                log_xi -= logsumexp(log_xi.reshape(len(seq) - 1, -1), axis=1)[:, None, None]
                start_acc += gamma[0]
                trans_acc += np.exp(log_xi).sum(axis=0)
                weight += gamma.sum(axis=0)
                mean_acc += gamma.T @ seq
                for state in range(self.n_states):
                    diff = seq - self.mean[state]
                    var_acc[state] += (gamma[:, state : state + 1] * (diff * diff)).sum(axis=0)
            self.start = start_acc / start_acc.sum()
            self.trans = trans_acc / np.maximum(trans_acc.sum(axis=1, keepdims=True), 1e-12)
            self.mean = mean_acc / np.maximum(weight[:, None], 1e-8)
            self.var = var_acc / np.maximum(weight[:, None], 1e-8) + 1e-2
        return self

    def predictive_loglik(self, values):
        log_emit = self._log_emit(values)
        log_trans = np.log(self.trans + 1e-16)
        log_filt = np.log(self.start + 1e-16)
        output = np.empty(len(values))
        for index in range(len(values)):
            log_pred = log_filt if index == 0 else logsumexp(log_filt[:, None] + log_trans, axis=0)
            joint = log_pred + log_emit[index]
            output[index] = logsumexp(joint)
            log_filt = joint - output[index]
        return output


def class_sequences(pack, mask, max_sequences=6, max_len=1_200, min_len=30, max_dt=90.0):
    flag = mask.copy()
    flag[1:] &= pack["dt"][1:] <= max_dt
    padded = np.r_[False, flag, False]
    change = np.diff(padded.astype(np.int8))
    starts = np.flatnonzero(change == 1)
    ends = np.flatnonzero(change == -1)
    order = np.argsort(-(ends - starts))
    sequences = []
    for index in order:
        if ends[index] - starts[index] < min_len:
            break
        stop = min(ends[index], starts[index] + max_len)
        sequences.append(kinematic_matrix(pack, slice(int(starts[index]), int(stop))))
        if len(sequences) >= max_sequences:
            break
    return sequences


def hmm_scores(pack, normal_model, anomaly_models):
    values = kinematic_matrix(pack)
    normal_log = normal_model.predictive_loglik(values)
    anomaly_log = np.vstack([model.predictive_loglik(values) for model in anomaly_models])
    return anomaly_log.max(axis=0) - normal_log


print("HMM")
started = time.perf_counter()
normal_seqs = class_sequences(fit, fit["valid"])
anomaly_seqs = class_sequences(fit, fit["anom"])
fast_seqs = class_sequences(fit, fit["anom"] & (fit["speed"] >= 8.0), min_len=20)
print(f"  штатных отрезков {len(normal_seqs)}, аномальных {len(anomaly_seqs)}, быстрых {len(fast_seqs)}")
if not normal_seqs or not anomaly_seqs:
    raise RuntimeError("Для HMM не нашлось отрезков обоих классов.")
iterations = 8
normal_model = DiagHMM(n_states=3, n_iter=iterations, seed=SEED).fit(normal_seqs)
anomaly_models = [DiagHMM(n_states=3, n_iter=iterations, seed=SEED + 1).fit(anomaly_seqs)]
if fast_seqs:
    anomaly_models.append(DiagHMM(n_states=2, n_iter=iterations, seed=SEED + 2).fit(fast_seqs))
fit_scores = hmm_scores(fit, normal_model, anomaly_models)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = hmm_scores(test, normal_model, anomaly_models)
print(f"  порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "HMM",
    test_scores > tau,
    time.perf_counter() - started,
    f"Гауссовские HMM, аномальных моделей {len(anomaly_models)}. Порог {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Фильтр Хэмпела

## Принцип

В симметричном окне по скорости точка сравнивается с робастной «нормой» (медиана и MAD); резкий выброс по скорости получает высокий score.

## Формализация

Окно $W_i=\{v_{i-h},\ldots,v_{i+h}\}$, $m_i=\mathrm{median}(W_i)$, $\mathrm{MAD}_i=\mathrm{median}_{v\in W_i}|v-m_i|$:

$$s_i = \frac{|v_i-m_i|}{1{,}4826\,\mathrm{MAD}_i+\varepsilon}.$$

Реализация — пакетная (центрированное окно, края дополняются крайним значением), не каузальный онлайн-вариант.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def hampel_scores(speed, half_window=10):
    padded = np.pad(speed, half_window, mode="edge")
    windows = np.lib.stride_tricks.sliding_window_view(padded, 2 * half_window + 1)
    center = np.median(windows, axis=1)
    mad = np.median(np.abs(windows - center[:, None]), axis=1)
    return np.abs(speed - center) / (1.4826 * mad + 1e-9)


print("Hampel")
started = time.perf_counter()
fit_scores = hampel_scores(fit["speed"])
tau, train_f1 = select_threshold(fit_scores, fit, extra=(3.0, 3.5, 5.0))
test_scores = hampel_scores(test["speed"])
print(f"  полуокно 10, порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "Hampel",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно 21 точка по скорости. Порог {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# DBSCAN

## Принцип

Точки в пространстве признаков кластеризуются по плотности; изолированные точки (шум, метка $-1$) считаются аномалиями. Абсолютные координаты не входят в признаки.

## Формализация

Вектор признаков $\mathbf{u}_i$ — робастно нормированные $(\Delta x,\Delta y, v, a, \Delta\psi, \log\mathrm{shape})$. $\varepsilon$-окрестность $N_\varepsilon(\mathbf{u}_i)$; core-point при $|N_\varepsilon|\ge \mathrm{MinPts}$. Бинарная метка: $\hat{a}_i=\mathbf{1}[c_i=-1]$.

**Подбор $\varepsilon$:** сетка по квантилям $k$-distance на обучении; выбор по MCC и операционному DL в полосе `MCC_BAND`. Кластеризация на тесте — по тестовому облаку признаков.

Непрерывный score (для диагностики): $s_i=\max(d_k(\mathbf{u}_i), d_{\mathrm{core}}(\mathbf{u}_i))$; большее $s_i$ — сильнее изоляция от плотного режима.


In [ ]:
from sklearn.cluster import DBSCAN


def density_matrix(pack):
    return local_motion_matrix(pack)


def dbscan_anomaly_scores(features, min_samples, labels=None, core_indices=None):
    """k-distance и расстояние до ближайшей core-точки. Больше — более аномально."""
    neighbors = NearestNeighbors(n_neighbors=min_samples, algorithm="kd_tree").fit(features)
    k_distance = neighbors.kneighbors(features)[0][:, -1]
    if core_indices is None or len(core_indices) == 0:
        return k_distance
    core_nn = NearestNeighbors(n_neighbors=1, algorithm="kd_tree").fit(features[core_indices])
    core_distance = core_nn.kneighbors(features)[0][:, 0]
    return np.maximum(k_distance, core_distance)


print("DBSCAN")
started = time.perf_counter()
center, scale = robust_params(density_matrix(fit)[fit["valid"]])
fit_x = apply_robust(density_matrix(fit), center, scale)
test_x = apply_robust(density_matrix(test), center, scale)
min_samples = 8
neighbors = NearestNeighbors(n_neighbors=min_samples, algorithm="kd_tree").fit(fit_x)
distances, _ = neighbors.kneighbors(fit_x)
k_distance = np.sort(distances[:, -1])
eps_grid = np.unique(np.quantile(k_distance, [0.5, 0.7, 0.85, 0.93, 0.97]))
records = []
for eps in eps_grid:
    labels = DBSCAN(eps=float(eps), min_samples=min_samples, algorithm="kd_tree", n_jobs=-1).fit_predict(fit_x)
    predicted = labels == -1
    record = operating_record(fit, predicted, eps)
    record["eps"] = float(eps)
    print(
        f"  eps {eps:.3f}: доля шума {predicted.mean():.1%}, "
        f"MCC {record['mcc']:.3f}, DL {fmt_dl_m(record['loss'])}, F1 {record['f1']:.3f}"
    )
    records.append(record)
chosen, best_mcc, band_size = pick_operating_point(records)
best_eps = chosen["eps"]
best_f1 = chosen["f1"]
print(
    f"  выбран eps {best_eps:.3f}: MCC {chosen['mcc']:.3f}, лучший MCC {best_mcc:.3f}, "
    f"в полосе {band_size}, DL {fmt_dl_m(chosen['loss'])}, F1 {best_f1:.3f}"
)
model = DBSCAN(eps=best_eps, min_samples=min_samples, algorithm="kd_tree", n_jobs=-1)
test_labels = model.fit_predict(test_x)
test_scores = dbscan_anomaly_scores(
    test_x,
    min_samples,
    labels=test_labels,
    core_indices=model.core_sample_indices_,
)
print(
    f"  медиана score: шум {np.median(test_scores[test_labels == -1]) if np.any(test_labels == -1) else float('nan'):.4g}, "
    f"кластер {np.median(test_scores[test_labels != -1]) if np.any(test_labels != -1) else float('nan'):.4g}"
)
remember_train_operating_point(chosen["loss"], best_f1, chosen["mcc"])
show_result(
    "DBSCAN",
    test_labels == -1,
    time.perf_counter() - started,
    (
        f"eps {best_eps:.3f}, min_samples {min_samples}. "
        "Score = max(k-distance, расстояние до core). "
        f"DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {best_f1:.3f}."
    ),
    test_scores,
)





# Local Outlier Factor

## Принцип

Сравнивается локальная плотность точки с плотностью её $k$ соседей; низкая относительная плотность — аномалия.

## Формализация

$\mathrm{reachdist}_k(x,y)=\max\{d_k(y),d(x,y)\}$, локальная достижимая плотность $\mathrm{lrd}_k$, фактор выброса

$$\mathrm{LOF}_k(x)=\frac{1}{|N_k(x)|}\sum_{y\in N_k(x)}\frac{\mathrm{lrd}_k(y)}{\mathrm{lrd}_k(x)}.$$

Признаки $(\Delta x,\Delta y, v, a, \Delta\psi)$ с робастной нормировкой по `move` обучения. LOF на тесте пересчитывается по тестовому набору; $\tau$ переносится с обучения.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def lof_matrix(pack):
    return np.column_stack([pack["dx"], pack["dy"], pack["speed"], pack["acc"], pack["dpsi"]])


def lof_scores(matrix, n_neighbors=35):
    model = LocalOutlierFactor(n_neighbors=n_neighbors, algorithm="kd_tree", n_jobs=-1)
    # Совпадающие координаты ломают оценку плотности, поэтому точки чуть разводятся.
    jittered = np.asarray(matrix, dtype=float) + np.random.default_rng(SEED).normal(0.0, 1e-8, size=np.shape(matrix))
    model.fit_predict(jittered)
    return -model.negative_outlier_factor_


print("LOF")
started = time.perf_counter()
lof_center, lof_scale = robust_params(lof_matrix(fit)[fit["valid"]])
fit_scores = lof_scores(apply_robust(lof_matrix(fit), lof_center, lof_scale))
tau, train_f1 = select_threshold(fit_scores, fit, extra=(1.5, 2.0))
test_scores = lof_scores(apply_robust(lof_matrix(test), lof_center, lof_scale))
print(f"  соседей 35, порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "LOF",
    test_scores > tau,
    time.perf_counter() - started,
    f"k = 35. Порог LOF {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Isolation Forest

## Принцип

Случайные деревья изолируют точки разрезами по признакам; аномалии изолируются за меньшее число разбиений.

## Формализация

Средняя глубина изоляции $E[h(x)]$, нормировка $c(n)=2H(n-1)-2(n-1)/n$:

$$s_i = 2^{-E[h(\mathbf{u}_i)]/c(n)} - \tfrac{1}{2},$$

где $\mathbf{u}_i$ — робастно нормированный вектор признаков движения. Лес обучается на обучающей выборке, оценка — на том же пространстве признаков.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def forest_matrix(pack):
    return local_motion_matrix(pack)


print("Isolation Forest")
started = time.perf_counter()
forest_center, forest_scale = robust_params(forest_matrix(fit)[fit["valid"]])
fit_x = apply_robust(forest_matrix(fit), forest_center, forest_scale)
test_x = apply_robust(forest_matrix(test), forest_center, forest_scale)
forest = IsolationForest(
    n_estimators=200,
    max_samples=256,
    contamination="auto",
    random_state=SEED,
    n_jobs=-1,
)
forest.fit(fit_x)
fit_scores = -forest.decision_function(fit_x)
tau, train_f1 = select_threshold(fit_scores, fit, extra=(0.0,))
test_scores = -forest.decision_function(test_x)
print(f"  порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "Isolation Forest",
    test_scores > tau,
    time.perf_counter() - started,
    f"200 деревьев, max_samples 256. Порог {tau:.3f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)



# One-Class SVM

## Принцип

Оценивается компактная область высокой плотности штатных признаков; точки вне области — аномалии.

## Формализация

$$\min_{\mathbf{w},\rho,\xi} \tfrac{1}{2}\|\mathbf{w}\|^2 + \frac{1}{\nu n}\sum_i \xi_i - \rho, \quad \mathbf{w}^\top\phi(\mathbf{u}_i)\ge\rho-\xi_i,\; \xi_i\ge0.$$

Решающая функция $f(\mathbf{u})=\sum_i \alpha_i K(\mathbf{u}_i,\mathbf{u})-\rho$; аномалия при $f(\mathbf{u})<0$. Score $s_i=-f(\mathbf{u}_i)$. Обучение — подвыборка `move` (до 4000 точек), ядро RBF.

**Подбор $\nu$:** сетка $\nu\in\{0{,}01,0{,}05,0{,}1\}$ по MCC и операционному DL на обучении. На тесте порог score фиксирован: $\hat{a}_i=\mathbf{1}[s_i>0]$.


In [ ]:
def svm_matrix(pack):
    return local_motion_matrix(pack)


print("One-Class SVM")
started = time.perf_counter()
svm_raw = svm_matrix(fit)
svm_center, svm_scale = robust_params(svm_raw[fit["valid"]])
normal_x = apply_robust(svm_raw[fit["valid"]], svm_center, svm_scale)
sample_size = min(len(normal_x), 4_000)
sample_index = rng.choice(len(normal_x), size=sample_size, replace=False)
train_x = normal_x[sample_index]
fit_x = apply_robust(svm_matrix(fit), svm_center, svm_scale)
test_x = apply_robust(svm_matrix(test), svm_center, svm_scale)
records = []
for nu in (0.01, 0.05, 0.1):
    model = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    model.fit(train_x)
    predicted = -model.decision_function(fit_x) > 0.0
    record = operating_record(fit, predicted, nu)
    record["model"] = model
    print(f"  nu {nu:.2f}: MCC {record['mcc']:.3f}, DL {fmt_dl_m(record['loss'])}, F1 {record['f1']:.3f}")
    records.append(record)
chosen, best_mcc, band_size = pick_operating_point(records)
best_model = chosen["model"]
best_f1 = chosen["f1"]
print(
    f"  выбран nu {chosen['tau']:.2f}: MCC {chosen['mcc']:.3f}, лучший MCC {best_mcc:.3f}, "
    f"в полосе {band_size}, DL {fmt_dl_m(chosen['loss'])}, F1 {best_f1:.3f}"
)
test_scores = -best_model.decision_function(test_x)
remember_train_operating_point(chosen["loss"], best_f1, chosen["mcc"])
show_result(
    "One-Class SVM",
    test_scores > 0.0,
    time.perf_counter() - started,
    (
        f"RBF, nu {chosen['tau']:.2f}, обучение на {nfmt(sample_size)} точках move. "
        f"Порог score фиксирован нулём. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {best_f1:.3f}."
    ),
    test_scores,
)





# Robust PCA

## Принцип

Матрица оконных признаков раскладывается на низкоранговую (типичное движение) и разреженную (редкие отклонения) компоненты; крупные элементы разреженной части — кандидаты в аномалии.

## Формализация

$$X = L + S, \qquad \min_{L,S}\|L\|_* + \lambda\|S\|_1.$$

Окна кинематических признаков длины 16, шаг 8; $\lambda=1/\sqrt{\max(n_1,n_2)}$. Разложение — inexact ALM (чередование SVD-порога и мягкого порога по $S$). Score точки $i$: $s_i=\max_{W:\,i\in W}\|S_W(i)\|_2$ по окнам, содержащим $i$.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def soft_threshold(matrix, amount):
    return np.sign(matrix) * np.maximum(np.abs(matrix) - amount, 0.0)


def inexact_alm_rpca(data, lam=None, max_iter=30, tol=1e-5):
    """Principal Component Pursuit: min ||L||_* + λ||S||_1 при X = L + S."""
    work = np.asarray(data, dtype=float)
    rows, cols = work.shape
    if lam is None:
        lam = 1.0 / np.sqrt(max(rows, cols))
    spectral = np.linalg.norm(work, 2)
    mu = 1.25 / max(spectral, 1e-8)
    mu_max = mu * 1e7
    factor = 1.6
    lagrange = work / max(spectral, np.max(np.abs(work)) / lam)
    sparse = np.zeros_like(work)
    low_rank = np.zeros_like(work)
    for _ in range(max_iter):
        basis, singular, right = np.linalg.svd(work - sparse + lagrange / mu, full_matrices=False)
        low_rank = (basis * np.maximum(singular - 1.0 / mu, 0.0)) @ right
        residual = work - low_rank + lagrange / mu
        sparse = soft_threshold(residual, lam / mu)
        error = work - low_rank - sparse
        lagrange = lagrange + mu * error
        if np.linalg.norm(error, "fro") / max(np.linalg.norm(work, "fro"), 1e-8) < tol:
            break
        mu = min(mu * factor, mu_max)
    return low_rank, sparse


def rpca_point_scores(pack, length=16, stride=8):
    features = np.column_stack([pack["dx"] / 1000.0, pack["dy"] / 1000.0, pack["speed"], pack["dpsi"]])
    starts = window_starts(pack, length, max_dt=180.0, max_dist=np.inf, require_normal=False)
    starts = starts[::stride]
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    index = starts[:, None] + np.arange(length)[None, :]
    matrix = features[index].reshape(len(starts), -1)
    center, scale = robust_params(matrix)
    _, sparse = inexact_alm_rpca(apply_robust(matrix, center, scale))
    row_score = np.linalg.norm(sparse.reshape(len(starts), length, -1), axis=2)
    return scatter_max(len(pack["lat"]), starts, row_score, length)


print("Robust PCA")
started = time.perf_counter()
fit_scores = rpca_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = rpca_point_scores(test)
print(f"  порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "Robust PCA",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окна длины 16, λ = 1/√max(n, m). Порог {tau:.3f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# LSTM-автоэнкодер

## Принцип

Сеть восстанавливает окна штатной локальной траектории; аномальное окно даёт большую ошибку восстановления.

## Формализация

Окно $X_i\in\mathbb{R}^{L\times 2}$ — координаты в км относительно первой точки окна. Энкодер–декодер: $\hat{X}_i=D(E(X_i))$.

**Оптимизация обучения:** $\min_{\theta,\phi}\sum_i \|X_i-\hat{X}_i\|_2^2$ (MSE, Adam, фиксированное число эпох).

На калибровочных штатных окнах оцениваются $\mu_j$, $\sigma_j^2$ компонент вектора ошибки. Score окна: $s_W=\sum_j (e_j-\mu_j)^2/\sigma_j^2$; точке $i$ присваивается $\max_{W\ni i} s_W$.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
class LSTMAutoencoder(nn.Module):
    """Encoder-decoder: вся последовательность восстанавливается из финального состояния."""

    def __init__(self, n_features=2, hidden=8):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.decoder = nn.LSTM(n_features, hidden, batch_first=True)
        self.proj = nn.Linear(hidden, n_features)

    def forward(self, sequence):
        _, state = self.encoder(sequence)
        decoded, _ = self.decoder(torch.zeros_like(sequence), state)
        return self.proj(decoded)


def relative_xy_km(pack, starts, length):
    index = starts[:, None] + np.arange(length)[None, :]
    block = pack["xy"][index]
    return ((block - block[:, :1, :]) / 1000.0).astype(np.float32)


def split_starts(starts, train_fraction=0.85):
    order = rng.permutation(len(starts))
    cut = max(1, int(len(starts) * train_fraction))
    return starts[order[:cut]], starts[order[cut:]]


def diagonal_mahalanobis(error, mean, var):
    return np.sum((error - mean) ** 2 / var, axis=1)


print("LSTM-автоэнкодер")
started = time.perf_counter()
lstm_length = 16
lstm_starts = window_starts(fit, lstm_length, max_dt=90.0, max_dist=2_000.0, require_normal=True)
lstm_starts = take_starts(lstm_starts, N_WINDOWS)
train_starts, calib_starts = split_starts(lstm_starts)
if len(calib_starts) < 10:
    calib_starts = train_starts[: max(10, len(train_starts) // 5)]
train_windows = np.clip(relative_xy_km(fit, train_starts, lstm_length), -30.0, 30.0)
calib_windows = relative_xy_km(fit, calib_starts, lstm_length)
lstm = LSTMAutoencoder()
train_torch(lstm, train_windows, lambda model, batch: ((model(batch) - batch) ** 2).mean(), N_EPOCHS)
calib_recon = batched_numpy(lstm, np.clip(calib_windows, -30.0, 30.0), lambda model, batch: model(batch))
calib_error = (calib_recon - calib_windows).reshape(len(calib_windows), -1)
error_mean = calib_error.mean(axis=0)
error_var = np.maximum(calib_error.var(axis=0), 1e-6)


def lstm_point_scores(pack):
    starts = window_starts(pack, lstm_length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    windows = relative_xy_km(pack, starts, lstm_length)
    recon = batched_numpy(lstm, np.clip(windows, -30.0, 30.0), lambda model, batch: model(batch))
    error = (recon - windows).reshape(len(starts), -1)
    window_score = diagonal_mahalanobis(error, error_mean, error_var)
    return scatter_max(len(pack["lat"]), starts, window_score, lstm_length)


fit_scores = lstm_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = lstm_point_scores(test)
print(f"  окон обучения {len(train_starts)}, порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "LSTM-автоэнкодер",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно {lstm_length}, скрытое состояние 8. Порог Махаланобиса {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Temporal Convolutional Network

## Принцип

Каузальная свёрточная сеть предсказывает следующую точку окна; аномалия — большой нормированный остаток прогноза.

## Формализация

По окну длины $L$ сетка $f_\theta$ предсказывает $\hat{\mathbf{x}}_{i}$ — следующую точку в локальных км. Остаток $\mathbf{r}_i=\mathbf{x}_i-\hat{\mathbf{x}}_i$.

**Оптимизация:** $\min_\theta \sum \|\mathbf{r}_i\|_2^2$ на штатных окнах обучения.

На калибровке: $\mu_c$, $\sigma_c^2$ по координатам остатка; $s_i=\sum_c (r_{i,c}-\mu_c)^2/\sigma_c^2$ для индекса прогноза (если окно не собрано, $s_i=0$).

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
class CausalTCN(nn.Module):
    """Каузальная свёрточная сеть: прогноз следующей точки по левому контексту."""

    def __init__(self, n_features=2, channels=16, levels=3, kernel=3):
        super().__init__()
        blocks = []
        pads = []
        in_channels = n_features
        dilation = 1
        for _ in range(levels):
            blocks.append(nn.Conv1d(in_channels, channels, kernel, dilation=dilation))
            pads.append((kernel - 1) * dilation)
            in_channels = channels
            dilation *= 2
        self.blocks = nn.ModuleList(blocks)
        self.pads = pads
        self.head = nn.Linear(channels, n_features)

    def forward(self, sequence):
        hidden = sequence.transpose(1, 2)
        for conv, pad in zip(self.blocks, self.pads):
            hidden = nn.functional.pad(hidden, (pad, 0))
            hidden = torch.relu(conv(hidden))
        return self.head(hidden[:, :, -1])


def tcn_pairs(pack, length, require_normal, limit=None):
    max_dist = 2_000.0 if require_normal else np.inf
    starts = window_starts(pack, length + 1, max_dt=90.0, max_dist=max_dist, require_normal=require_normal)
    if limit is not None:
        starts = take_starts(starts, limit)
    history = relative_xy_km(pack, starts, length)
    origin = pack["xy"][starts]
    target = ((pack["xy"][starts + length] - origin) / 1000.0).astype(np.float32)
    return starts, history, target


print("TCN")
started = time.perf_counter()
tcn_length = 16
pair_starts, history, target = tcn_pairs(fit, tcn_length, require_normal=True, limit=N_WINDOWS)
order = rng.permutation(len(pair_starts))
cut = max(1, int(len(order) * 0.85))
train_idx, calib_idx = order[:cut], order[cut:]


def train_predictor(model, features, targets, epochs):
    """Учит модель, которая по окну предсказывает вектор той же размерности, что и target."""
    features = np.clip(np.ascontiguousarray(features), -30.0, 30.0).astype(np.float32)
    targets = np.clip(np.ascontiguousarray(targets), -30.0, 30.0).astype(np.float32)
    loader = DataLoader(
        TensorDataset(torch.from_numpy(features), torch.from_numpy(targets)),
        batch_size=256,
        shuffle=True,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            loss = ((model(batch_x) - batch_y) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch_x)
            seen += len(batch_x)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


tcn = CausalTCN()
train_predictor(tcn, history[train_idx], target[train_idx], N_EPOCHS)
with torch.no_grad():
    calib_pred = batched_numpy(tcn, np.clip(history[calib_idx], -30.0, 30.0), lambda model, batch: model(batch))
residual = calib_pred - target[calib_idx]
residual_mean = residual.mean(axis=0)
residual_var = np.maximum(residual.var(axis=0), 1e-6)


def tcn_point_scores(pack):
    starts, hist, truth = tcn_pairs(pack, tcn_length, require_normal=False)
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    pred = batched_numpy(tcn, np.clip(hist, -30.0, 30.0), lambda model, batch: model(batch))
    score = diagonal_mahalanobis(pred - truth, residual_mean, residual_var)
    point_score = np.zeros(len(pack["lat"]))
    np.maximum.at(point_score, starts + tcn_length, score)
    return point_score


fit_scores = tcn_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = tcn_point_scores(test)
print(f"  окон обучения {len(train_idx)}, порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "TCN",
    test_scores > tau,
    time.perf_counter() - started,
    f"Каузальное окно {tcn_length}. Порог Махаланобиса остатка {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# DONUT

## Принцип

Вариационный автоэнкодер сжимает окно траектории в латентное пространство и восстанавливает его; окна с низкой правдоподобностью под моделью — аномалии.

## Формализация

$q_\phi(z|x)=\mathcal{N}(\mu_\phi(x),\mathrm{diag}(\sigma_\phi^2(x)))$, декодер $p_\theta(x|z)$. ELBO: $\mathcal{L}=\mathbb{E}_{q}[\log p_\theta(x|z)] - D_{\mathrm{KL}}(q_\phi(z|x)\|p(z))$.

**Оптимизация:** $\max_{\theta,\phi}\mathcal{L}$ (в коде минимизируется MSE реконструкции плюс KL, Adam).

Score окна: $s_W=\|x-\hat{x}\|_2^2 + D_{\mathrm{KL}}$ при реконструкции через $\mu_\phi$; $s_i=\max_{W\ni i}s_W$.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
class WindowVAE(nn.Module):
    """Полносвязный VAE для одного окна, записанного одним вектором."""

    def __init__(self, dim, latent=8):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.mu = nn.Linear(32, latent)
        self.logvar = nn.Linear(32, latent)
        self.decoder = nn.Sequential(
            nn.Linear(latent, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim),
        )

    def forward(self, flat, sample):
        hidden = self.encoder(flat)
        mu = self.mu(hidden)
        logvar = self.logvar(hidden).clamp(-8.0, 8.0)
        if sample:
            latent = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        else:
            latent = mu
        return self.decoder(latent), mu, logvar


def vae_loss(model, batch):
    recon, mu, logvar = model(batch, sample=True)
    mse = ((recon - batch) ** 2).mean()
    kl = -0.5 * torch.mean(1.0 + logvar - mu.pow(2) - logvar.exp())
    return mse + kl


def vae_window_table(pack, starts, length, mean, std):
    windows = relative_xy_km(pack, starts, length).reshape(len(starts), -1)
    scaled = (windows - mean) / std
    return np.clip(scaled, -30.0, 30.0).astype(np.float32)


print("DONUT")
started = time.perf_counter()
vae_length = 16
vae_starts = take_starts(window_starts(fit, vae_length, max_dt=90.0, max_dist=2_000.0, require_normal=True), N_WINDOWS)
raw_windows = relative_xy_km(fit, vae_starts, vae_length).reshape(len(vae_starts), -1)
vae_mean = raw_windows.mean(axis=0)
vae_std = np.where(raw_windows.std(axis=0) < 1e-6, 1.0, raw_windows.std(axis=0))
vae_train = np.clip((raw_windows - vae_mean) / vae_std, -8.0, 8.0).astype(np.float32)
vae = WindowVAE(vae_train.shape[1])
train_torch(vae, vae_train, vae_loss, N_EPOCHS)


def donut_point_scores(pack):
    starts = window_starts(pack, vae_length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    table = vae_window_table(pack, starts, vae_length, vae_mean, vae_std)

    def score_batch(model, batch):
        recon, mu, logvar = model(batch, sample=False)
        mse = ((recon - batch) ** 2).sum(dim=-1)
        kl = -0.5 * torch.sum(1.0 + logvar - mu.pow(2) - logvar.exp(), dim=-1)
        return mse + kl

    window_score = batched_numpy(vae, table, score_batch)
    return scatter_max(len(pack["lat"]), starts, window_score, vae_length)


fit_scores = donut_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = donut_point_scores(test)
print(f"  порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "DONUT",
    test_scores > tau,
    time.perf_counter() - started,
    f"VAE, окно {vae_length}, латентность 8. Порог −ELBO {tau:.2f}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Anomaly Transformer

## Принцип

В окне последовательности совмещаются ошибка восстановления и расхождение матрицы внимания с гауссовым prior по индексам позиций.

## Формализация

В слое $\ell$: ассоциации $A^{(\ell)}_{ij}=\mathrm{softmax}(Q_iK_j^\top/\sqrt{d})$, prior $P^{(\ell)}_{ij}\propto\exp(-(i-j)^2/(2\sigma_\ell^2))$. Association discrepancy $\mathrm{AssDis}_i$ — симметричный KL между строками $A$ и $P$. Итог позиции $i$ в окне:

$$\mathrm{AnomalyScore}_i = \mathrm{softmax}(-\mathrm{AssDis})_i \cdot \|\mathbf{x}_i-\hat{\mathbf{x}}_i\|_2^2.$$

**Оптимизация:** $\min \|X-\hat{X}\|_2^2$ (ошибка реконструкции). Вход: $(\Delta x/1000,\Delta y/1000, v, \Delta\psi)$ с нормировкой по штатным окнам. $s_i=\max_{W\ni i}\mathrm{AnomalyScore}_i$.

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
def symmetric_kl(attention, prior):
    """Симметричный KL по последней оси. prior имеет форму (L, L) и растягивается на батч."""
    prior = prior.unsqueeze(0)
    log_gap_left = torch.log(attention.clamp_min(1e-8)) - torch.log(prior.clamp_min(1e-8))
    log_gap_right = torch.log(prior.clamp_min(1e-8)) - torch.log(attention.clamp_min(1e-8))
    return 0.5 * ((attention * log_gap_left).sum(-1) + (prior * log_gap_right).sum(-1))


class AnomalyTransformer(nn.Module):
    """Уменьшенный Anomaly Transformer: внимание, prior-ассоциация и ошибка восстановления."""

    def __init__(self, n_features=4, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        self.input = nn.Linear(n_features, d_model)
        self.attention = nn.ModuleList(
            [nn.MultiheadAttention(d_model, n_heads, batch_first=True) for _ in range(n_layers)]
        )
        self.norm_attn = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.feed = nn.ModuleList(
            [
                nn.Sequential(nn.Linear(d_model, d_model * 2), nn.ReLU(), nn.Linear(d_model * 2, d_model))
                for _ in range(n_layers)
            ]
        )
        self.norm_feed = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_layers)])
        self.output = nn.Linear(d_model, n_features)
        self.raw_sigma = nn.Parameter(torch.zeros(n_layers))

    def forward(self, sequence):
        hidden = self.input(sequence)
        length = sequence.shape[1]
        index = torch.arange(length, device=sequence.device)
        distance2 = (index[:, None] - index[None, :]).float().pow(2)
        discrepancy = []
        for layer, attn in enumerate(self.attention):
            updated, weights = attn(hidden, hidden, hidden, need_weights=True, average_attn_weights=True)
            hidden = self.norm_attn[layer](hidden + updated)
            hidden = self.norm_feed[layer](hidden + self.feed[layer](hidden))
            sigma = torch.nn.functional.softplus(self.raw_sigma[layer]) + 1.0
            prior = torch.softmax(-distance2 / (2.0 * sigma**2), dim=-1)
            discrepancy.append(symmetric_kl(weights, prior))
        association = torch.stack(discrepancy, dim=0).mean(dim=0)
        recon = self.output(hidden)
        point_error = (recon - sequence).pow(2).sum(dim=-1)
        score = torch.softmax(-association, dim=-1) * point_error
        return recon, score


def transformer_windows(pack, starts, length, mean, std):
    index = starts[:, None] + np.arange(length)[None, :]
    step = np.stack(
        [pack["dx"][index] / 1000.0, pack["dy"][index] / 1000.0, pack["speed"][index], pack["dpsi"][index]],
        axis=-1,
    )
    scaled = (step - mean) / std
    return np.clip(scaled, -30.0, 30.0).astype(np.float32)


print("Anomaly Transformer")
started = time.perf_counter()
at_length = 32
at_starts = take_starts(window_starts(fit, at_length, max_dt=90.0, max_dist=2_000.0, require_normal=True), N_WINDOWS)
at_index = at_starts[:, None] + np.arange(at_length)[None, :]
at_raw = np.stack(
    [
        fit["dx"][at_index] / 1000.0,
        fit["dy"][at_index] / 1000.0,
        fit["speed"][at_index],
        fit["dpsi"][at_index],
    ],
    axis=-1,
)
at_mean = at_raw.mean(axis=(0, 1))
at_std = at_raw.std(axis=(0, 1))
at_std = np.where(at_std < 1e-6, 1.0, at_std)
at_train = np.clip((at_raw - at_mean) / at_std, -8.0, 8.0).astype(np.float32)
transformer = AnomalyTransformer()


def transformer_loss(model, batch):
    recon, _ = model(batch)
    return ((recon - batch) ** 2).mean()


train_torch(transformer, at_train, transformer_loss, N_EPOCHS)


def transformer_point_scores(pack):
    starts = window_starts(pack, at_length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    starts = starts[::2]
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    windows = transformer_windows(pack, starts, at_length, at_mean, at_std)
    scores = batched_numpy(transformer, windows, lambda model, batch: model(batch)[1])
    return scatter_max(len(pack["lat"]), starts, scores, at_length)


fit_scores = transformer_point_scores(fit)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = transformer_point_scores(test)
print(f"  порог {tau:.3g}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "Anomaly Transformer",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно {at_length}, 2 слоя, d_model 32. Порог {tau:.3g}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# STGVAD

## Принцип

В окне траектории строится граф «прошлое → настоящее» с пространственными рёбрами между близкими точками; сеть предсказывает относительное положение точки по сообщениям соседей.

## Формализация

Вершина $i$ несёт $(v_i/10, \Delta\psi_i)$; сосед $j<i$ — $(\Delta x_j,\Delta y_j, v_j, \Delta\psi_j)$ относительно начала окна. Ребро $j\to i$ при $d_{\mathrm{geo}}(p_i,p_j)<5$ км; строки матрицы смежности нормированы. Прогноз $\hat{\mathbf{r}}_i$ положения в км; $s_i=1000\|\hat{\mathbf{r}}_i-\mathbf{r}_i\|$ (для первой точки окна $s=0$). $s_i=\max_{W\ni i}$ по покрывающим окнам.

**Оптимизация:** $\min \|\hat{\mathbf{r}}_{i}-\mathbf{r}_{i}\|_2^2$ по точкам окна, кроме первой (MSE, Adam).

Бинарная метка аномалии: $\hat{a}_i = \mathbf{1}[\hat{y}_i = 0]$. Если модель выдаёт скалярный score $s_i$ (большее значение — сильнее подозрение на аномалию), то $\hat{a}_i = \mathbf{1}[s_i > \tau]$. Порог $\tau$ (или гиперпараметр вроде $\varepsilon$, $\nu$) подбирается на обучающей выборке: максимум MCC, при равенстве в полосе `MCC_BAND` — минимум операционного distance loss.


In [ ]:
class STGNN(nn.Module):
    """Два слоя message passing. Собственные координаты точки в расчёт не входят."""

    def __init__(self):
        super().__init__()
        self.own1 = nn.Linear(2, 32)
        self.msg1 = nn.Linear(4, 32)
        self.own2 = nn.Linear(32, 32)
        self.msg2 = nn.Linear(32, 32)
        self.head = nn.Linear(32, 2)

    def forward(self, own, neigh, adj):
        hidden = torch.relu(self.own1(own) + self.msg1(torch.matmul(adj, neigh)))
        hidden = torch.relu(self.own2(hidden) + self.msg2(torch.matmul(adj, hidden)))
        return self.head(hidden)


def graph_batch(pack, starts, length):
    index = starts[:, None] + np.arange(length)[None, :]
    xy = pack["xy"][index]
    relative = xy - xy[:, :1, :]
    distance = np.linalg.norm(relative[:, :, None, :] - relative[:, None, :, :], axis=-1)
    past = np.tri(length, k=-1, dtype=np.float32)
    adj = ((distance < 5_000.0) & (distance > 0.0)).astype(np.float32) * past
    adj /= np.maximum(adj.sum(axis=-1, keepdims=True), 1.0)
    speed = (pack["speed"][index] / 10.0).astype(np.float32)
    turn = pack["dpsi"][index].astype(np.float32)
    own = np.clip(np.stack([speed, turn], axis=-1), -30.0, 30.0).astype(np.float32)
    neigh = np.clip(
        np.stack([relative[:, :, 0] / 1000.0, relative[:, :, 1] / 1000.0, speed, turn], axis=-1),
        -30.0,
        30.0,
    ).astype(np.float32)
    target = (relative / 1000.0).astype(np.float32)
    return own, neigh, adj.astype(np.float32), target


def train_stgnn(model, own, neigh, adj, target, epochs):
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(own),
            torch.from_numpy(neigh),
            torch.from_numpy(adj),
            torch.from_numpy(np.clip(target, -30.0, 30.0)),
        ),
        batch_size=128,
        shuffle=True,
    )
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        total, seen = 0.0, 0
        for batch_own, batch_neigh, batch_adj, batch_y in loader:
            batch_own = batch_own.to(DEVICE)
            batch_neigh = batch_neigh.to(DEVICE)
            batch_adj = batch_adj.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(batch_own, batch_neigh, batch_adj)
            loss = ((pred[:, 1:, :] - batch_y[:, 1:, :]) ** 2).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total += float(loss.detach()) * len(batch_own)
            seen += len(batch_own)
        print(f"    эпоха {epoch + 1}/{epochs}, loss {total / max(seen, 1):.5f}")
    model.eval()
    return model


def stgnn_point_scores(model, pack, length):
    starts = window_starts(pack, length, max_dt=120.0, max_dist=np.inf, require_normal=False)
    starts = starts[::2]
    if len(starts) == 0:
        return np.zeros(len(pack["lat"]))
    pieces = []
    batch = 512
    for cursor in range(0, len(starts), batch):
        own, neigh, adj, target = graph_batch(pack, starts[cursor : cursor + batch], length)
        with torch.no_grad():
            pred = (
                model(
                    torch.from_numpy(own).to(DEVICE),
                    torch.from_numpy(neigh).to(DEVICE),
                    torch.from_numpy(adj).to(DEVICE),
                )
                .cpu()
                .numpy()
            )
        error = np.linalg.norm(pred - target, axis=-1) * 1000.0
        error[:, 0] = 0.0
        pieces.append(error)
    return scatter_max(len(pack["lat"]), starts, np.concatenate(pieces, axis=0), length)


print("STGVAD")
started = time.perf_counter()
graph_length = 16
graph_starts = take_starts(
    window_starts(fit, graph_length, max_dt=90.0, max_dist=2_000.0, require_normal=True),
    N_WINDOWS,
)
own_fit, neigh_fit, adj_fit, target_fit = graph_batch(fit, graph_starts, graph_length)
stgnn = train_stgnn(STGNN(), own_fit, neigh_fit, adj_fit, target_fit, N_EPOCHS)
fit_scores = stgnn_point_scores(stgnn, fit, graph_length)
tau, train_f1 = select_threshold(fit_scores, fit)
test_scores = stgnn_point_scores(stgnn, test, graph_length)
print(f"  порог {fmt_distance_m(tau)}, DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}")
show_result(
    "STGVAD",
    test_scores > tau,
    time.perf_counter() - started,
    f"Окно {graph_length}, пространственный радиус 5 км. Порог {fmt_distance_m(tau, 1)}. DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {train_f1:.3f}.",
    test_scores,
)


# Максимальная физически согласованная траектория

## Принцип

Жадный якорный алгоритм: точка принимается как `move`, если от последней принятой она достижима с допустимой скоростью и формой окна; иначе — аномалия, якорь не сдвигается.

## Формализация

Якорь $j$ — последняя принятая точка ($\hat{a}_j=0$). Для $i>j$:

$$v_{ij}=\frac{d_{\mathrm{geo}}(p_j,p_i)}{\max(t_i-t_j,\,1)}, \qquad \hat{a}_i=\mathbf{1}\bigl[v_{ij}>v_+ \;\lor\; \mathrm{shape}_i>\tau_{\mathrm{shape}}\bigr].$$

При $\hat{a}_i=0$ якорь переходит в $i$. **Подбор** $(v_+,\tau_{\mathrm{shape}})$: сетка по MCC и операционному DL.

Непрерывный score при отклонении:

$$s_i=\max(0,v_{ij}-v_+)+\max(0,\mathrm{shape}_i-\tau_{\mathrm{shape}})+d_{\mathrm{geo}}(p_j,p_i)/1000; \quad s_i=0 \text{ для принятой точки}.$$


In [ ]:
def mpct_detect(pack, v_max, shape_limit):
    """Аномалия, если точка недостижима от последней принятой или окно слишком извилистое.

    Якорь — последняя точка, прошедшая оба порога. Отклонённая точка якорь не заменяет.
    """
    lat = pack["lat"]
    lon = pack["lon"]
    t_sec = pack["t"]
    shape = pack["shape"]
    count = len(x_m)
    predicted = np.zeros(count, dtype=bool)
    scores = np.zeros(count, dtype=float)
    last = 0
    for index in range(1, count):
        distance = float(haversine_m(lat[last], lon[last], lat[index], lon[index]))
        delta = max(float(t_sec[index] - t_sec[last]), 1.0)
        speed = distance / delta
        shape_excess = max(0.0, float(shape[index]) - shape_limit)
        if speed > v_max or shape[index] > shape_limit:
            predicted[index] = True
            scores[index] = max(0.0, speed - v_max) + shape_excess + distance / 1000.0
        else:
            last = index
    return predicted, scores


print("MPCT")
started = time.perf_counter()
records = []
for speed_limit in (8.0, 10.0, 12.0, 15.0, 20.0):
    for shape_limit in (1.5, 2.0, 3.0):
        predicted, _ = mpct_detect(fit, speed_limit, shape_limit)
        record = operating_record(fit, predicted, speed_limit)
        record["speed"] = float(speed_limit)
        record["shape_limit"] = float(shape_limit)
        print(
            f"  v_max {speed_limit:.0f} м/с, форма {shape_limit:.1f}: "
            f"доля исключённых {predicted.mean():.1%}, "
            f"MCC {record['mcc']:.3f}, DL {fmt_dl_m(record['loss'])}, F1 {record['f1']:.3f}"
        )
        records.append(record)
chosen, best_mcc, band_size = pick_operating_point(records)
best_speed = chosen["speed"]
best_shape = chosen["shape_limit"]
best_f1 = chosen["f1"]
print(
    f"  выбрано v_max {best_speed:.0f} м/с, форма {best_shape:.1f}: "
    f"MCC {chosen['mcc']:.3f}, лучший MCC {best_mcc:.3f}, "
    f"в полосе {band_size}, DL {fmt_dl_m(chosen['loss'])}, F1 {best_f1:.3f}"
)
test_pred, test_scores = mpct_detect(test, best_speed, best_shape)
print(
    f"  медиана score: вне маршрута {np.median(test_scores[test_pred]) if np.any(test_pred) else 0:.4g}, "
    f"на маршруте {np.median(test_scores[~test_pred]) if np.any(~test_pred) else 0:.4g}"
)
remember_train_operating_point(chosen["loss"], best_f1, chosen["mcc"])
show_result(
    "MPCT",
    test_pred,
    time.perf_counter() - started,
    (
        f"Якорь — последняя принятая точка. "
        f"v_max {best_speed:.0f} м/с, форма окна ≤ {best_shape:.1f}. "
        f"DL обучения {fmt_dl_m(LAST_TRAIN_DL)}, F1 {best_f1:.3f}."
    ),
    test_scores,
)




# Сводка

Таблица собрана из `TrajectoryMetrics.compute_all` на тестовой выборке. Модели упорядочены по MCC: он растёт, когда одновременно растут и доля пойманных аномалий, и доля сохранённого штатного хода. При близком MCC выше строка с меньшим distance loss. Постоянные разметки в этот порядок не входят: они показывают края шкалы.

Положительный класс метрик классификации — аномалия, метка `0`. Геопространственные метрики (RMSE, max(d_i), distance loss) считаются после линейной интерполяции координат в аномальных точках. Тепловая карта показывает precision, recall и F1. Столбцы «DL обуч.», «F1 обуч.» и «MCC обуч.» — качество выбранного порога на обучении; DL обуч. — операционный, без интерполяции. DL и DLR в таблице на тесте — из `TrajectoryMetrics` (с интерполяцией аномалий).



In [ ]:
summary = pd.DataFrame(RESULTS)
if summary.empty:
    raise RuntimeError("Нет результатов: сводка собирается после ячеек моделей.")
if "mcc" not in summary.columns:
    raise RuntimeError("В результатах нет MCC: заново выполните ячейки моделей.")

ref_m = report_reference_route_length(test)
baselines = ["Всегда штатная", "Всегда аномалия"]
is_baseline = summary["Метод"].isin(baselines)
models = summary.loc[~is_baseline].copy()


def finite_column(frame, key, missing):
    values = frame[key].to_numpy(dtype=float)
    return np.where(np.isfinite(values), values, missing)


models["_mcc"] = -finite_column(models, "mcc", -1.0)
models["_dl"] = finite_column(models, "distance_loss_m", 1e18)
models["_f1"] = -finite_column(models, "f1", -1.0)
models = models.sort_values(["_mcc", "_dl", "_f1", "Метод"], kind="mergesort")
baseline_rows = summary.loc[is_baseline].copy()
ordered = pd.concat([models, baseline_rows], ignore_index=True)

table_columns = [
    "Метод",
    "mcc",
    "distance_loss_m",
    "distance_loss_ratio",
    "f1",
    "precision",
    "recall",
    "rmse_m",
    "max_deviation_m",
    "delay_median",
    "DL обучения, км",
    "F1 обучения",
    "MCC обучения",
    "Время, с",
]
printable = ordered[table_columns].copy()
printable["distance_loss_m"] = printable["distance_loss_m"].apply(
    lambda v: meters_to_km(v) if np.isfinite(v) else v
)
headers = {
    "mcc": "MCC",
    "distance_loss_m": "DL, км",
    "distance_loss_ratio": "DLR",
    "f1": "F1",
    "precision": "Precision",
    "recall": "Recall",
    "rmse_m": "RMSE, м",
    "max_deviation_m": "Max(d_i), м",
    "delay_median": "Delay, с",
    "DL обучения, км": "DL обуч., км",
    "F1 обучения": "F1 обуч.",
    "MCC обучения": "MCC обуч.",
    "Время, с": "Время, с",
}
printable = printable.rename(columns=headers)
print(f"Эталон маршрута на тесте (отчётный): {fmt_route_km(ref_m)}.")
print("Порядок строк: сначала MCC. При близком MCC — меньший distance loss, затем F1.")
print(
    printable.to_string(
        index=False,
        float_format=lambda value: "—" if not np.isfinite(value) else (fmt_num(value, 4) if abs(float(value)) >= 1000 else f"{float(value):.4f}"),
    )
)
if ref_m == 0.0:
    print(
        "Эталон на тесте нулевой: в тестовом отрезке нет точек move. "
        "Distance loss равен длине пути, который модель оставила классом move."
    )

rate_columns = [key for key, _ in RATE_FIELDS]
heat = np.ma.masked_invalid(ordered[rate_columns].to_numpy(dtype=float, copy=True))
fig, ax = plt.subplots(figsize=(12.4, 0.42 * len(ordered) + 1.8))
image = ax.imshow(heat, cmap="YlOrRd", vmin=0.0, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(RATE_FIELDS)))
ax.set_xticklabels([label for _, label in RATE_FIELDS], rotation=30, ha="right")
ax.set_yticks(range(len(ordered)))
ax.set_yticklabels(ordered["Метод"])
for row in range(heat.shape[0]):
    for col in range(heat.shape[1]):
        value = heat[row, col]
        label = "—" if np.ma.is_masked(value) else f"{float(value):.3f}"
        ax.text(col, row, label, ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
ax.set_title("Доли от 0 до 1. Строки упорядочены по MCC")
fig.tight_layout()
try:
    from IPython import get_ipython

    interactive = get_ipython() is not None
except Exception:
    interactive = False
if interactive:
    plt.show()
else:
    plt.close(fig)


def fmt_rate(value):
    return "—" if not np.isfinite(value) else f"{float(value):.4f}"


print()
print("Методы по MCC, без постоянных разметок:")
for _, item in models.iterrows():
    print(
        f"  {item['Метод']:<24} MCC {fmt_rate(item['mcc'])}   "
        f"DL {fmt_dl_m(item['distance_loss_m'])}   "
        f"F1 {fmt_rate(item['f1'])}"
    )
leader = models.iloc[0]
print()
print(
    f"По разделению классов впереди {leader['Метод']}: "
    f"MCC {fmt_rate(leader['mcc'])}, DL {fmt_dl_m(leader['distance_loss_m'])}, "
    f"F1 {fmt_rate(leader['f1'])}."
)



# Выводы по прогону

Фрагмент `[200000:350000)`, без точек `stand` (их отброшено 6 842): обучение 93 158 точек (45% `anomaly`, 55% `move`), тест 50 000 (77% `anomaly`, 23% `move`). Эталон маршрута на тесте — 560 км. Если ничего не вырезать, distance loss равен 121 млн м (DLR 216): сырая ломаная длиннее эталона примерно в 217 раз. На тесте 7 аномальных событий. DBSCAN на обучающем облаке остановился с `MemoryError` и в сводку не вошёл.

Две шкалы здесь расходятся. MCC и point-wise F1 отвечают, узнана ли точка. Distance loss отвечает, какую ломаную модель оставила как маршрут. На этом тесте аномалий 77%, поэтому разметка «все точки аномальны» даёт F1 0,87 при MCC 0 и пустом маршруте: DL равен эталону, 560 км, DLR 1. F1 в такой доле классов хвалит уничтожение штатного хода. Порог и сводка идут от MCC, а длина сравнивается уже среди моделей, которые различают классы.

## Кто разделяет классы

По MCC на тесте впереди Isolation Forest: MCC 0,86, F1 0,97, полнота 0,96, точность 0,98, PR-AUC 0,99. Сохранено 92% точек `move`. Все 7 событий найдены, средняя задержка 4 с. DL 15,3 млн м (DLR 27), RMSE 24 км, Hausdorff 49 км. Лес обучен на обучающей выборке и тем же лесом размечает тест. Порог −0,18 стоит у медианы штатного score (−0,19; у аномалий −0,10). На обучении MCC этого порога 0,82.

Anomaly Transformer: MCC 0,77, F1 0,94, полнота 0,90, точность 0,98, сохранено 92% `move`. По точкам это второй результат, и MCC обучения совпал с тестом, 0,77. По длине маршрута остаётся пропуск: DL 88 млн м, DLR 157, Hausdorff 7 497 км — та же верхняя точка, что у разметки «ничего не вырезать». Найдено 6 из 7 событий.

One-Class SVM (`nu` 0,10, 4 000 точек `move`): MCC 0,71, F1 0,92, полнота 0,87, сохранено 91% `move`. Все 7 событий, средняя задержка 1 с. DL 32 млн м (DLR 57), Hausdorff 459 км. MCC обучения 0,80.

Дальше по точкам идут Robust PCA и HMM. Robust PCA: MCC 0,69, F1 0,90, полнота 0,83, сохранено 95% `move`, DLR 160, Hausdorff 7 497 км, 6 из 7 событий, медианная задержка 15 с. HMM: MCC 0,65, F1 0,88, полнота 0,80, сохранено 95% `move`, DLR 81, Hausdorff 460 км, все 7 событий, средняя задержка 4 с, PR-AUC 0,99.

## Длина маршрута

MPCT (`v_max` 10 м/с, форма окна ≤ 1,5): MCC 0,64, F1 0,89, точность 0,97, полнота 0,82, сохранено 91% `move`. Все 7 событий, средняя задержка 11 с. По точкам он шестой. По длине это лучший результат прогона: DL 1,90 млн м, DLR 3,4, RMSE 12 км, Hausdorff 23 км. Ошибка составляет несколько эталонов. На обучении у этой пары порогов MCC 0,84 при лучшем MCC сетки 0,85; внутри полосы 0,02 взята пара с меньшим distance loss.

Фильтр Калмана даёт следующую разумную длину: MCC 0,57, полнота 0,74, точность 0,97, сохранено 93% `move`, DL 9,4 млн м (DLR 17), RMSE 64 км, Hausdorff 459 км, все 7 событий, средняя задержка 1 с. Медианы score — 0,0002 у `move` и 11 у аномалий. Выбранный порог 0,007 ниже теоретического χ²₀.₉₉₉ = 13,8.

Isolation Forest по длине следующий среди моделей с MCC выше 0,5: DLR 27. У One-Class SVM DLR 57. Дальше рабочие модели оставляют DLR от 69 до 168.

## Где метка точки и маршрут расходятся

У Anomaly Transformer, Robust PCA, STGVAD, DONUT, LSTM и TCN Hausdorff один и тот же, 7 497 км: в принятом маршруте остаётся хотя бы один дальний скачок. Все шесть находят 6 из 7 событий. Доля сохранённого `move` у них 91–98%. Полнота точек разная: Anomaly Transformer 0,90, STGVAD 0,69, DONUT 0,61, LSTM 0,58, TCN 0,46. У DONUT и LSTM медианная задержка 625 с. MCC обучения у оконных сетей 0,65–0,77. На тесте STGVAD, DONUT, LSTM и TCN опускаются до 0,51, 0,48, 0,43 и 0,38. Anomaly Transformer переносит порог по MCC и оставляет в маршруте глобальный скачок.

ARIMA сохраняет 97% `move` при точности 0,98, полноте 0,54 и MCC 0,43. DL 42 млн м. События она видит (affiliation F1 0,96, все 7), точки внутри эпизода пропускает. Коэффициенты сняты с гладкого куска в 6 503 точки.

GP-EKF разделяет score: медиана `move` 0,41, у аномалий 363. Порог 11, выбранный по MCC на обучении (там MCC 0,62), на тесте даёт MCC 0,36, полноту 0,57 и сохраняет 85% `move`. DL 39 млн м (DLR 69). Все 7 событий найдены, средняя задержка 100 с. До Калмана и MPCT по длине и до Isolation Forest по точкам ему далеко.

Фильтр частиц сохраняет 96% `move` при полноте 0,36, MCC 0,29 и DLR 121.

Hampel и LOF классы почти не разделяют. Медианы score: Hampel 0,58 у `move` и 0,66 у аномалий, LOF 1,044 и 1,047. Полнота 0,14 и 0,12, MCC 0,12 и 0,003, DLR 138 и 133.

Гауссовский процесс по шагу ставит более высокий score штатному ходу: медиана 0,63 у `move` против 0,12 у аномалий. Порог 8,1 помечает аномалией 2% теста, полнота 0,03, штатный ход сохранён целиком, MCC 0,08. События числятся найденными за счёт позднего касания: средняя задержка 2 186 с.

## Что следует из этого

Для точечной разметки опорный Isolation Forest: лучший MCC и F1, почти все аномалии найдены, 92% штатного хода сохранено, лес строится по обучению. Плата — DLR 27.

Для длины маршрута опорная модель — MPCT. Он удерживает оба класса (91% штатного хода и 82% аномалий) и ошибку длины в 3,4 эталона. Следом по длине идёт фильтр Калмана, DLR 17, при MCC 0,57.

DBSCAN в этом прогоне результата не дал. Гауссовский процесс, Hampel и LOF к разметке этого теста не годятся: первый почти не срабатывает при шкале, где штатный ход выглядит страннее аномалии, последние два не отделяют классы по локальной скорости и плотности.
